# Lab 5. QAOA — 좋은 분할을 찾고, 출력 분포를 평가하기

## 실습 소개

이번 Lab에서는 <strong>그래프의 정점들을 두 그룹으로 나누어 높은 점수를 얻는 분할을 찾습니다.</strong> 양자 회로의 측정 비트열을 분할 하나로 해석하고, 각도를 바꾸어 평균 점수를 높입니다. 탐색을 마치면 평균 점수와 함께 가장 좋은 답을 얼마나 자주 얻는지도 확인합니다.

Lab4와 마찬가지로 <strong>상태 준비 → 평가 → 각도 갱신</strong>을 반복합니다. 이번에는 에너지 대신 그래프의 분할 점수를 사용하므로, 먼저 답을 어떻게 표현하고 평가할지 정해야 합니다.

그래프는 대상인 정점과 두 정점 사이의 연결인 간선으로 이루어집니다. 이번 정점과 간선은

$$
V=\{0,1,2,3\},
\qquad \mathcal E=\{(0,1),(1,2),(2,3),(3,0),(0,2)\}
$$

입니다. 정점을 순서대로 사각형의 꼭짓점에 놓으면 네 변과 대각선 하나에 해당합니다. 간선에는 방향이 없으므로 $(3,0)$과 $(0,3)$은 같은 연결이며 한 번만 셉니다.

목표는 <strong>서로 다른 그룹을 잇는 간선을 최대한 많이 만드는 것</strong>입니다. 이를 ‘잘린 간선’이라고 부르지만 실제로 선을 삭제한다는 뜻은 아닙니다. 잘린 간선 하나에 1점을 주고 합계를 최대화하는 문제가 <strong>MaxCut</strong>입니다. 예를 들어 정점 0, 1을 한 그룹에, 2, 3을 다른 그룹에 놓으면 $(1,2),(3,0),(0,2)$가 잘려 점수는 3입니다.

각 정점에 그룹을 나타내는 비트 하나를 붙이면 분할을 비트열로 나타낼 수 있습니다. 정점 $i$는 큐비트 qi에 대응시키며, 문자열은 <strong>$z_3z_2z_1z_0$ 순서</strong>로 읽습니다. 오른쪽 끝이 정점 0입니다. 앞의 분할에서 정점 0, 1을 그룹 1로 두면 문자열은 `0011`이 됩니다. 한 그룹이 비어 있는 경우도 후보에 포함하므로 가능한 비트열은 16개입니다. 자세한 읽기와 점수 계산은 문제 1에서 연습합니다.

회로를 반복 실행하면 서로 다른 비트열이 나올 수 있습니다. 비트열 $z$의 출력 확률이 $P(z)$, 점수가 $C(z)$라면 평균 점수는

$$
\langle C\rangle=\sum_z P(z)C(z)
$$

입니다. 이번에는 이 평균을 높이는 회로 각도를 찾습니다.

회로는 어느 분할도 먼저 선호하지 않도록 균등 중첩 $|+\rangle^{\otimes4}$에서 시작합니다. 이어 <strong>cost 연산</strong>으로 비트열의 점수에 따른 위상을 붙이고, <strong>mixer 연산</strong>으로 성분들을 결합합니다. 위상만 바꾸면 Z 측정 확률은 그대로이지만, 뒤의 결합 과정에서는 상대위상이 진폭의 덧셈과 상쇄에 영향을 줄 수 있습니다. 따라서 두 연산을 연결한 뒤 실제 분포를 평가해야 합니다.

이처럼 점수에 따른 연산과 mixer를 번갈아 적용하고 고전 최적화기로 각도를 찾는 방법을 <strong>QAOA(Quantum Approximate Optimization Algorithm)</strong>라고 합니다. 이번에는 cost 각도 $\gamma$, mixer 각도 $\beta$를 사용해

$$
|\psi(\gamma,\beta)\rangle
=U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$

를 준비합니다. 오른쪽부터 읽어 균등 중첩, cost, mixer 순서로 적용합니다. 이 묶음을 한 번 사용하는 경우를 <strong>$p=1$</strong>이라고 부르며, $p$는 전체 게이트 수가 아닙니다. 각 연산의 정의와 회전각을 게이트로 옮기는 과정은 문제 2에서 차례로 설명합니다.

어떤 각도에서나 점수가 좋아지는 것은 아닙니다. 제공 최적화기는 각도를 바꾸며 평가 결과를 비교하고, 최소화하는 함수에는 평균 점수의 음수 $-\langle C\rangle$를 전달합니다. 찾은 각도는 다시 회로에 넣어 검증합니다.

결과에서는 <strong>평균 점수와 최적해 확률을 구분</strong>합니다. 평균에는 모든 답의 점수가 반영되지만, 최적해 확률은 최대 점수를 얻는 모든 비트열의 확률을 합한 값입니다. 평균이 더 높은 분포가 최적해도 더 자주 낸다고 단정할 수는 없습니다. Lab4의 유일한 바닥상태와 달리 최적 비트열은 여러 개일 수 있으므로, 답 하나의 확률만으로 성능을 평가하지 않겠습니다.

이번 문제는 작아서 16개 후보를 전수조사할 수 있습니다. 그 결과는 회로 출력을 평가할 기준으로 사용하고, 회로 자체는 일반적인 입력 각도에서 주어진 QAOA 정의를 구현해야 합니다.

| 문제 | 확인할 내용 |
|---|---|
| <strong>1. 분할의 점수와 비교 기준 만들기</strong> | 비트열을 분할로 읽고 점수 함수와 고전 기준표를 만듭니다. |
| <strong>2. 점수를 회로의 연산으로 옮기기</strong> | 점수 관측량과 cost·mixer를 연결해 회로를 작성합니다. |
| <strong>3. 각도 탐색과 회로의 역할 확인하기</strong> | 최적화하고, 연산을 빼거나 순서를 바꾼 회로를 비교합니다. |
| <strong>4. 출력 분포의 성능 평가하기</strong> | 평균과 최적해 확률, 정확값과 표본 결과를 구분합니다. |

마지막에는 모든 간선의 점수를 두 배로 바꾸고, 원래 출력 분포를 유지하려면 각도를 어떻게 조절할지 판단합니다. 점수의 단위가 달라지는 것과 더 좋은 답을 얻는 것은 같은 뜻인지도 확인하겠습니다.

---

## 노트북 사용 방법

각 절은 계산의 목적과 기호 정의, 필요한 수식과 예제, 직접 풀 문제로 이어집니다. <strong>분할의 점수, 그 점수를 나타내는 관측량, 상태를 바꾸는 회로 연산</strong>을 구분하며 읽으세요.

코드 셀은 위에서 아래로 실행합니다. ‘제공 코드’에는 비트열 읽기, 상태·확률 계산, 최적화, 측정 기록 생성과 표 출력 도구가 들어 있으므로 이 셀도 실행합니다. 내부 구현을 모두 분석할 필요는 없으며 필요한 사용법은 각 문제에서 안내합니다.

1. <strong>계산과 예상을 먼저 작성합니다.</strong> 그래프와 점수, 초기 상태, 연산 순서와 각도를 확인하고 결과를 예상합니다.
2. <strong>TODO를 완성하고 검사합니다.</strong> `TODO BEGIN`과 `TODO END` 사이를 작성하되 함수 이름과 매개변수는 유지합니다. 답안 셀 다음에 공개 확인 셀을 실행하세요.
3. <strong>관찰을 켜고 해당 실험을 실행합니다.</strong> 학생용의 `RUN_OBSERVATIONS`는 처음에 `False`입니다. 예상을 적은 뒤 `True`로 바꾸고 설정 셀과 해당 관찰 셀을 실행하세요. 다음 문항도 결과를 열기 전에 먼저 예상합니다. 관찰을 다시 막으려면 `False`로 바꾸고 설정 셀을 실행합니다. 공개 함수 검사는 이 설정과 무관하게 작동합니다.
4. <strong>관찰 결과를 해석합니다.</strong> 얻은 비트열이 어떤 분할인지 확인하고, 한 번의 답과 분포 전체의 성능을 구분합니다. 코드를 수정했다면 답안·검사·연결 실험을 다시 실행하고, 회로가 바뀌었으면 이전 최적화 결과도 다시 계산하세요.

| 작성할 함수 | 담당하는 일 |
|---|---|
| `cut_score` | 비트열 하나의 분할 점수를 계산합니다. |
| `cost_layer` | 점수에 따른 위상 연산을 구현합니다. |
| `qaoa_p1` | 초기 상태 준비, cost, mixer를 연결합니다. |
| `distribution_metrics` | 출력 기록에서 평균 점수와 최적해 확률을 계산합니다. |

최적화기와 목적함수는 제공됩니다. 학생 함수는 여러 입력에서 작동해야 하며, 특정 비트열이나 각도의 결과만 직접 반환하지 않습니다. 마지막 적용은 예상값과 제안 각도를 입력한 뒤 별도의 `RUN_APPLICATION`을 켜서 확인합니다.

회전각은 <strong>라디안</strong>입니다. 상태벡터와 문자열은 $q_3q_2q_1q_0$ 순서이며, 정점 번호를 읽을 때 `vertex_bit` 도구를 사용할 수 있습니다. 각 간선은 방향 없는 연결로 한 번씩만 셉니다.

앞부분에서는 이상적인 상태의 <strong>정확한 확률과 기댓값</strong>을 사용합니다. 뒤에서는 새 상태를 정해진 횟수만큼 준비해 Z 측정한 비트열 기록을 얻습니다. 이 반복 횟수가 <strong>shots</strong>입니다. 각 기록의 점수를 평균하면 평균 점수의 추정값, 최적해 기록의 비율을 구하면 최적해 확률의 추정값이 됩니다. 유한한 표본에는 변동이 있으므로 정확값과 추정값을 구분하세요. Counts는 전체 횟수로 정규화하며, 나타나지 않은 비트열의 횟수는 0입니다.

확인 셀의 <strong>미완성</strong>은 남은 TODO, <strong>실패</strong>는 실행 오류 또는 조건 불일치, <strong>통과</strong>는 공개 코드 검사 통과를 뜻합니다. 회로 검사는 정의된 상태를 준비하는지 확인하고, 성능 평가는 그 상태가 얼마나 좋은 분할을 내는지 확인합니다. 둘은 다른 질문이므로 서술 답안에도 어떤 조건과 지표를 비교했는지 적으세요.

이 Lab에서는 <strong>AI를 활용해 공부해도 좋습니다.</strong> 개념 설명, 수식 점검, 디버깅에 활용하되 제안받은 코드도 직접 실행하세요. 설명 문항에는 실제 관찰 결과를 근거로 점수 계산과 회로 구성, 성능 평가가 왜 적절한지 자신의 말로 작성합니다.

## 실행 환경 준비

이 노트북은 Python 3.12, Qiskit 2.3.1, NumPy 2.2.6, SciPy 1.18.1에서 검증했습니다. 로컬에서는 함께 배포된 `requirements.txt`를 설치한 환경을 Jupyter 커널로 선택하세요.

패키지가 없다면 새 코드 셀에서 다음 명령을 실행한 뒤 커널을 다시 시작하고, 노트북을 위에서부터 실행하세요.

```python
%pip install qiskit==2.3.1 numpy==2.2.6 scipy==1.18.1
```

아래 제공 도구 셀도 실행합니다. 긴 코드는 접혀 보일 수 있습니다. 각 문제에서 도구의 역할과 사용법을 안내하므로 내부 구현을 모두 분석한 뒤 시작할 필요는 없습니다.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy", "scipy"):
    print(package, version(package))

### 제공 도구와 관찰 실행 설정

아래 도구 셀을 실행하세요. 공개 확인 셀은 학습용 검사이며 서술 답안은 별도로 작성합니다. 관찰 결과를 보기 전에는 각 문항의 예상과 계산을 먼저 적습니다.

학생용은 `RUN_OBSERVATIONS=False`로 시작합니다. 관찰할 준비가 되면 아래 설정을 `True`로 바꾸어 실행하고, 해당 문항의 관찰 실험 셀을 실행하세요. 이 설정은 함수 TODO가 아닙니다. 마지막 적용에는 별도의 실행 설정이 있습니다.

In [ ]:
"""제공 코드: 4정점 MaxCut의 연산자·상태·표본·최적화·학습용 검사."""
import numbers
from html import escape
from types import CodeType, FunctionType, ModuleType
from contextlib import contextmanager
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.circuit import Gate, Instruction
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector, state_fidelity

N_VERTICES = 4
EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (0, 2))
BITSTRINGS = tuple(format(index, '04b') for index in range(16))
ATOL = 1e-9
CHECK_LABELS = ('1b · 분할 점수', '2b · cost 연산', '2c · QAOA 한 층', '4b · 분포 지표')
CHECK_STATUS = {}
LAB_RESULTS = {}
INITIAL_STATE = Statevector(np.ones(16, dtype=complex) / 4)


def show_table(headers, rows):
    def table_text(value):
        if isinstance(value, (float, np.floating)):
            return f'{value:.8f}'
        if isinstance(value, (tuple, list)):
            return '(' + ', '.join(table_text(item) for item in value) + ')'
        return str(value)
    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError('표의 열 수를 확인하세요.')
    header = ''.join(f'<th>{escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(table_text(x))}</td>' for x in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def _real(value, label='반환값'):
    if not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_)) or not np.isfinite(value):
        raise AssertionError(f'{label}: 유한한 실수가 필요합니다.')
    return float(value)


def _scale(value):
    value = _real(value, 'cost_scale')
    if value <= 0:
        raise ValueError('cost_scale은 양수입니다.')
    return value


def _integer_score(value):
    if not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)):
        raise AssertionError('cut_score는 정수를 반환해야 합니다.')
    return int(value)


def vertex_bit(bits, vertex):
    return int(bits[-1 - vertex])


def _pauli_word(letter, vertices):
    word = ['I'] * N_VERTICES
    for vertex in vertices:
        word[N_VERTICES - 1 - vertex] = letter
    return ''.join(word)


def cost_operator(cost_scale=1.0):
    scale = _scale(cost_scale)
    return SparsePauliOp.from_list([('IIII', scale * len(EDGES) / 2)] +
                                  [(_pauli_word('Z', edge), -scale / 2) for edge in EDGES])


MIXER_OPERATOR = SparsePauliOp.from_list([(_pauli_word('X', (q,)), 1.0) for q in range(N_VERTICES)])


def definition_cost_matrix(gamma, cost_scale=1.0):
    diagonal = np.diag(cost_operator(cost_scale).to_matrix()).real
    return np.diag(np.exp(-1j * _real(gamma, 'gamma') * diagonal))


def definition_state(gamma, beta, cost_scale=1.0):
    """공개된 지수 연산의 정의를 직접 계산하는 독립 기준입니다."""
    phase = np.diag(definition_cost_matrix(gamma, cost_scale)) * INITIAL_STATE.data
    mixer = expm(-1j * _real(beta, 'beta') * MIXER_OPERATOR.to_matrix())
    return Statevector(np.einsum('ij,j->i', mixer, phase))


@contextmanager
def _instrument_dependencies(function, replacements):
    """실제 호출 경로의 함수·별칭을 잠시 계측하고 모든 참조를 복원합니다."""
    if not isinstance(function, FunctionType):
        raise AssertionError('제공된 함수 정의 안에 답안을 작성하세요.')
    namespaces, functions, originals = {}, {}, {}
    library_prefixes = ('numpy', 'scipy', 'qiskit', 'IPython', 'importlib', 'contextlib')

    def code_names(code):
        names = set(code.co_names)
        for constant in code.co_consts:
            if isinstance(constant, CodeType):
                names.update(code_names(constant))
        return names

    def dependency(value, name=None):
        key = name if name in replacements else getattr(value, '__name__', None)
        if isinstance(value, FunctionType) and key in replacements:
            originals[id(value)] = (value, replacements[key])
            return True
        return False

    def visit(candidate):
        if not isinstance(candidate, FunctionType) or id(candidate) in functions:
            return
        if dependency(candidate):
            return
        if (candidate.__module__ or '').startswith(library_prefixes):
            return
        functions[id(candidate)] = candidate
        namespace = candidate.__globals__
        namespaces[id(namespace)] = namespace
        for name in replacements:
            dependency(namespace.get(name), name)
        names = code_names(candidate.__code__)
        for name in names:
            value = namespace.get(name)
            if isinstance(value, FunctionType):
                if not dependency(value, name):
                    visit(value)
            elif isinstance(value, ModuleType) and not value.__name__.startswith(library_prefixes):
                module_namespace = vars(value)
                namespaces[id(module_namespace)] = module_namespace
                for attribute in names:
                    target = module_namespace.get(attribute)
                    if not dependency(target, attribute):
                        visit(target)
        for value in (*tuple(candidate.__defaults__ or ()),
                      *tuple((candidate.__kwdefaults__ or {}).values())):
            if not dependency(value):
                visit(value)
        for cell in candidate.__closure__ or ():
            try:
                value = cell.cell_contents
            except ValueError:
                continue
            if not dependency(value):
                visit(value)

    visit(function)
    saved_bindings, saved_defaults, saved_cells = [], [], []
    seen_cells = set()

    def substituted(value):
        match = originals.get(id(value))
        return match[1] if match is not None and value is match[0] else value

    try:
        for namespace in namespaces.values():
            for name, value in tuple(namespace.items()):
                replacement = substituted(value)
                if replacement is not value:
                    saved_bindings.append((namespace, name, value))
                    namespace[name] = replacement
        for candidate in functions.values():
            defaults, keyword_defaults = candidate.__defaults__, candidate.__kwdefaults__
            new_defaults = None if defaults is None else tuple(substituted(value) for value in defaults)
            new_keywords = None if keyword_defaults is None else {
                name: substituted(value) for name, value in keyword_defaults.items()}
            if (defaults is not None and any(a is not b for a, b in zip(defaults, new_defaults))) or (
                    keyword_defaults is not None and any(keyword_defaults[key] is not new_keywords[key] for key in keyword_defaults)):
                saved_defaults.append((candidate, defaults, keyword_defaults))
                candidate.__defaults__, candidate.__kwdefaults__ = new_defaults, new_keywords
            for cell in candidate.__closure__ or ():
                if id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                try:
                    value = cell.cell_contents
                except ValueError:
                    continue
                replacement = substituted(value)
                if replacement is not value:
                    saved_cells.append((cell, value))
                    cell.cell_contents = replacement
        yield
    finally:
        for cell, value in reversed(saved_cells):
            cell.cell_contents = value
        for candidate, defaults, keyword_defaults in reversed(saved_defaults):
            candidate.__defaults__, candidate.__kwdefaults__ = defaults, keyword_defaults
        for namespace, name, value in reversed(saved_bindings):
            namespace[name] = value


def _check_circuit_operations(circuit, allowed=None, active=None):
    """허용된 기본 게이트와 그 게이트만 묶은 사용자 정의 회로를 검사합니다."""
    forbidden = {'measure', 'reset', 'initialize', 'set_statevector', 'set_density_matrix',
                 'if_else', 'while_loop', 'for_loop', 'switch_case'}
    active = set() if active is None else active
    if id(circuit) in active:
        raise AssertionError('순환하는 합성 게이트 정의는 사용할 수 없습니다.')
    active.add(id(circuit))
    try:
        if circuit.num_clbits:
            raise AssertionError('상태 준비 회로에는 고전 비트를 넣지 마세요.')
        for item in circuit.data:
            operation = item.operation
            if operation.name in forbidden or operation.num_clbits:
                raise AssertionError('측정·초기화·reset·조건부 제어 없이 유니터리 게이트를 사용하세요.')
            if allowed is not None and operation.name in allowed:
                continue
            # to_gate/to_instruction으로 묶은 회로만 펼칩니다. 다른 기본 게이트의
            # 분해를 이용해 이 문항의 허용 게이트 조건을 우회하지 않습니다.
            if operation.base_class in (Gate, Instruction) and operation.definition is not None:
                _check_circuit_operations(operation.definition, allowed, active)
            elif allowed is not None:
                raise AssertionError('해당 함수에 허용된 기본 게이트로 회로를 작성하세요. 합성 게이트도 내부를 검사합니다.')
    finally:
        active.remove(id(circuit))


def circuit_unitary(circuit, kind=None):
    if not isinstance(circuit, QuantumCircuit):
        raise AssertionError('QuantumCircuit을 반환하세요.')
    if (circuit.num_qubits, circuit.num_clbits) != (4, 0) or circuit.parameters:
        raise AssertionError('수치 각도가 정해진 4큐비트·고전 비트 0개 회로가 필요합니다.')
    allowed = {'cost': {'rzz', 'rz', 'cx', 'barrier'},
               'qaoa': {'h', 'rzz', 'rx', 'rz', 'cx', 'barrier'}}
    _check_circuit_operations(circuit, None if kind is None else allowed[kind])
    matrix = np.asarray(Operator(circuit).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(16), atol=ATOL, rtol=0):
        raise AssertionError('유한한 유니터리 회로가 필요합니다.')
    return matrix


def circuit_state(circuit, kind=None):
    return Statevector(circuit_unitary(circuit, kind)[:, 0].copy())


def probabilities_from_circuit(circuit):
    probabilities = np.asarray(circuit_state(circuit).probabilities(), dtype=float)
    probabilities /= probabilities.sum()
    return dict(zip(BITSTRINGS, map(float, probabilities)))


def mean_cut_from_circuit(circuit, cost_scale=1.0):
    return float(np.real(circuit_state(circuit).expectation_value(cost_operator(cost_scale))))


def classical_reference(cost_scale=1.0, score_function=None):
    """전체 16개 기준표. 기본은 현재 cut_score, 모듈 호출에서는 명시적으로 전달할 수 있습니다."""
    scale = _scale(cost_scale)
    function = score_function if score_function is not None else globals().get('cut_score')
    if not callable(function):
        raise NotImplementedError('먼저 cut_score를 작성하고 실행하세요.')
    scores = {bits: scale * _integer_score(function(bits)) for bits in BITSTRINGS}
    maximum = max(scores.values())
    optimal = tuple(bits for bits in BITSTRINGS if scores[bits] == maximum)
    return scores, maximum, optimal


def weight_vector(weights):
    if not isinstance(weights, dict) or not weights:
        raise ValueError('비어 있지 않은 비트열 가중치 dict가 필요합니다.')
    vector = np.zeros(16)
    for bits, value in weights.items():
        if bits not in BITSTRINGS:
            raise ValueError('비트열은 q3q2q1q0 순서의 네 비트입니다.')
        weight = _real(value, '가중치')
        if weight < 0:
            raise ValueError('가중치는 음이 아니어야 합니다.')
        vector[int(bits, 2)] = weight
    if vector.sum() <= 0:
        raise ValueError('전체 가중치 합은 양수여야 합니다.')
    return vector / vector.sum()


def sample_counts(circuit, shots=1024, seed=None):
    if not isinstance(shots, (int, np.integer)) or isinstance(shots, (bool, np.bool_)) or shots <= 0:
        raise ValueError('shots는 양의 정수입니다.')
    probabilities = weight_vector(probabilities_from_circuit(circuit))
    counts = np.random.default_rng(seed).multinomial(int(shots), probabilities)
    return {bits: int(value) for bits, value in zip(BITSTRINGS, counts) if value}


def optimize_qaoa(objective, initial=(0.1, 0.1)):
    initial = np.array([_real(value, '초기 각도') for value in initial], dtype=float)
    if initial.shape != (2,):
        raise ValueError('초기 각도는 gamma,beta 두 값입니다.')
    history = []
    def observed(parameters):
        value = _real(objective(np.asarray(parameters).copy()), '목적함수 값')
        history.append({'parameters': tuple(map(float, parameters)), 'objective_value': value, 'mean_cut': -value})
        return value
    result = minimize(observed, initial, method='Nelder-Mead',
                      options={'maxiter': 250, 'xatol': 1e-8, 'fatol': 1e-9})
    return {'parameters': tuple(map(float, result.x)), 'initial': tuple(map(float, initial)),
            'objective_value': float(result.fun), 'mean_cut': -float(result.fun),
            'initial_mean': history[0]['mean_cut'], 'nfev': int(result.nfev), 'nit': int(result.nit),
            'success': bool(result.success), 'message': str(result.message), 'history': history}


def _same_operator(actual, expected):
    overlap = np.vdot(expected.ravel(), actual.ravel())
    if abs(overlap) < ATOL:
        return False
    phase = overlap / abs(overlap)
    return bool(np.allclose(actual, phase * expected, atol=ATOL, rtol=0))


def _run_check(label, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[label] = '미완성'
        print(f'{label}: 미완성 — TODO를 작성한 뒤 다시 실행하세요.')
        return False
    except Exception as error:
        CHECK_STATUS[label] = '실패'
        print(f'{label}: 실패 — {type(error).__name__}: {error}')
        return False
    CHECK_STATUS[label] = '통과'
    print(f'{label}: 통과')
    return True


def check_cut_score(function):
    def check():
        diagonal = np.diag(cost_operator().to_matrix()).real
        for bits, expected in zip(BITSTRINGS, diagonal):
            assert _integer_score(function(bits)) == expected, '비트 위치와 간선 목록을 확인하세요.'
    return _run_check(CHECK_LABELS[0], check)


def check_cost_layer(function):
    def check():
        for gamma, scale in ((0.0, 1.0), (0.61, 1.0), (-0.47, 0.4), (1.07, 2.0)):
            actual = circuit_unitary(function(gamma, cost_scale=scale), 'cost')
            expected = definition_cost_matrix(gamma, scale)
            assert _same_operator(actual, expected), '전체 cost 연산의 상대위상·부호·배율을 확인하세요. 공통 전역위상은 허용합니다.'
    return _run_check(CHECK_LABELS[1], check)


def _check_cost_connection(function):
    """실제 답안은 유지한 채 cost_layer 반환 회로의 사용을 검사합니다."""
    if not isinstance(function, FunctionType):
        raise AssertionError('제공된 함수 정의 안에 회로를 작성하세요.')
    point = (0.37, -0.26, 1.7)
    def neutral_layer(angle):
        layer = QuantumCircuit(4)
        layer.rzz(0.179 * angle, 0, 3)
        layer.rz(-0.413 * angle, 1)
        layer.rzz(0.29 * angle, 1, 2)
        return layer
    probe = neutral_layer(point[0] * point[2])
    calls = []
    def cost(gamma, cost_scale=1.0):
        effective_angle = _real(gamma, 'gamma') * _scale(cost_scale)
        calls.append((gamma, cost_scale))
        return neutral_layer(effective_angle)
    with _instrument_dependencies(function, {'cost_layer': cost}):
        actual = circuit_state(function(point[0], point[1], cost_scale=point[2]), 'qaoa')
        after_cost = np.einsum('ij,j->i', circuit_unitary(probe), INITIAL_STATE.data)
        mixer = expm(-1j * point[1] * MIXER_OPERATOR.to_matrix())
        expected = Statevector(np.einsum('ij,j->i', mixer, after_cost))
        assert calls, '자신의 cost_layer를 호출하여 회로를 연결하세요.'
        assert state_fidelity(actual, expected) >= 1 - ATOL, 'cost_layer가 반환한 연산을 초기 준비 뒤, mixer 앞에 사용하세요.'
    return calls



def check_qaoa_p1(function):
    def check():
        for gamma, beta, scale in ((0.0, 0.0, 1.0), (0.38, 0.21, 1.0), (-0.73, 0.42, 2.0), (1.17, -0.28, 0.4)):
            actual = circuit_state(function(gamma, beta, cost_scale=scale), 'qaoa')
            expected = definition_state(gamma, beta, scale)
            assert state_fidelity(actual, expected) >= 1 - ATOL, '초기 상태, cost→mixer 순서와 각도·배율을 확인하세요.'
        _check_cost_connection(function)
    return _run_check(CHECK_LABELS[2], check)


def check_distribution_metrics(function):
    def check():
        cases = [
            ({'0011': 6, '0000': 2}, 2.25, 0.0),
            ({'0011': 2, '0000': 1, '0101': 3, '1010': 5}, 38/11, 8/11),
            ({'0001': 0.2, '1000': 0.3, '1010': 0.5}, 3.2, 0.5),
            ({bits: 1 for bits in BITSTRINGS}, 2.5, 0.125),
            ({'0101': 0, '0011': 7}, 3.0, 0.0),
        ]
        for weights, base_mean, expected_optimal in cases:
            for scale in (1e-10, 0.4, 1.0, 2.0):
                answer = function(dict(weights), cost_scale=scale)
                assert isinstance(answer, tuple) and len(answer) == 2, '(mean_cut,p_opt) tuple을 반환하세요.'
                mean, optimal = (_real(value) for value in answer)
                assert np.isclose(mean, scale * base_mean, atol=ATOL, rtol=0), '전체 가중치로 정규화하고 점수 배율을 적용하세요.'
                assert np.isclose(optimal, expected_optimal, atol=ATOL, rtol=0), '전체 16개 후보의 최적해 집합에 속하는 가중치를 모두 합하세요.'
    return _run_check(CHECK_LABELS[3], check)


def run_demo(label, experiment, requires=()):
    LAB_RESULTS.pop(label, None)
    missing = [key for key in requires if CHECK_STATUS.get(key) != '통과']
    if missing:
        print(f'[대기] {label}: 먼저 다음 코드 확인을 통과하세요: {", ".join(missing)}')
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f'[대기] {label}: 앞의 TODO를 완성하세요.')
        return None
    except Exception as error:
        print(f'[실험 오류] {label}: {type(error).__name__}: {error}')
        return None
    LAB_RESULTS[label] = result
    return result

In [ ]:
RUN_OBSERVATIONS = False
print("관찰 실험 실행:", RUN_OBSERVATIONS)
print("먼저 해당 문항의 예상과 계산을 적은 뒤 True로 바꾸고 관찰 셀을 실행하세요.")

## 문제 1. 그래프의 분할에 어떻게 점수를 매길까요?

이번 문제에서는 <strong>비트열 하나가 나타내는 분할을 읽고, 그 분할의 점수를 계산하는 함수</strong>를 만듭니다. 이어서 가능한 모든 비트열을 조사해 최대 점수와 최적해 집합을 구합니다.

이 결과는 뒤에서 양자 회로의 출력을 평가하는 기준이 됩니다. 회로가 비트열을 출력하더라도, 그 비트열의 점수를 올바르게 계산하지 못하면 회로의 성능도 잘못 판단하게 됩니다. 따라서 먼저 정점 번호, 비트 위치, 간선의 점수를 정확하게 연결하겠습니다.

---

### 비트열 하나가 두 그룹을 정합니다

이번 그래프의 정점은 $0,1,2,3$이고, 간선은 다음과 같습니다.

$$
\mathcal E
=
\{(0,1),(1,2),(2,3),(3,0),(0,2)\}.
$$

각 정점 $i$에 비트 $z_i$를 하나씩 대응시킵니다. 비트가 0이면 그룹 0, 1이면 그룹 1에 속한다고 정하겠습니다.

| 정점의 비트 | 정점이 속하는 그룹 |
|---|---|
| $z_i=0$ | 그룹 0 |
| $z_i=1$ | 그룹 1 |

각 정점은 두 그룹 중 하나에만 속합니다. 모든 정점의 비트를 정하면 분할 하나가 정해집니다. 한 그룹이 비어 있는 경우도 후보에 포함합니다.

두 그룹의 이름 자체에는 우열이 없습니다. 그룹 0과 그룹 1의 이름을 서로 바꾸어도, 어떤 정점들이 같은 그룹에 속하는지는 변하지 않습니다.

이름을 바꾸는 일을 비트열로 나타내면 모든 0을 1로, 모든 1을 0으로 바꾸는 것입니다. 예를 들어 `0011`과 `1100`은 그룹의 이름만 바꾼 같은 분할을 나타냅니다. 두 문자열이 측정 기록에서는 서로 다른 결과로 나타난다는 점도 함께 기억해 두세요.

---

### 문자열의 위치와 정점 번호 맞추기

Qiskit의 네 비트 문자열은

$$
z_3z_2z_1z_0
$$

순서입니다. 오른쪽 끝의 비트가 정점 0, 왼쪽 끝의 비트가 정점 3에 해당합니다.

예를 들어 문자열 `0011`은 다음과 같이 읽습니다.

| 정점 $i$ | 문자열에서 읽은 $z_i$ | 속하는 그룹 |
|---:|---:|---|
| 0 | 1 | 그룹 1 |
| 1 | 1 | 그룹 1 |
| 2 | 0 | 그룹 0 |
| 3 | 0 | 그룹 0 |

따라서 이 문자열이 나타내는 두 그룹은

$$
G_0=\{2,3\},
\qquad
G_1=\{0,1\}
$$

입니다. 여기서 $G_0,G_1$은 각각 그룹 0과 그룹 1에 속한 정점들의 집합입니다.

코드에서는 문자열의 위치를 직접 계산하는 대신, 제공되는 다음 도구를 사용할 수 있습니다.

```python
vertex_bit(bits, vertex)
```

이 함수는 문자열 `bits`에서 정점 `vertex`의 비트를 읽어 정수 0 또는 1을 반환합니다. 예를 들어 `vertex_bit("0011", 0)`은 1, `vertex_bit("0011", 3)`은 0입니다.

이 도구가 문자열의 방향을 처리해 주므로, 점수 함수를 작성할 때에는 <strong>간선의 양 끝점이 같은 그룹인지 다른 그룹인지</strong>에 집중하면 됩니다.

---

### 한 간선의 점수는 두 끝점의 그룹으로 정합니다

간선 $(i,j)$의 양 끝점이 서로 다른 그룹에 속하면 그 간선의 점수를 1, 같은 그룹에 속하면 0으로 정합니다.

$$
c_{ij}(z)
=
\begin{cases}
1,&z_i\ne z_j,\\
0,&z_i=z_j.
\end{cases}
$$

전체 분할의 점수는 다섯 간선의 점수를 더한 값입니다.

$$
C(z)=\sum_{(i,j)\in\mathcal E}c_{ij}(z).
$$

이번에는 모든 간선의 가중치가 1이므로, $C(z)$는 잘린 간선의 수와 같습니다.

간선에 방향은 없습니다. $(i,j)$와 $(j,i)$는 같은 연결을 나타냅니다. 제공된 간선 목록에는 각 간선이 한 번씩 들어 있으므로, 목록의 각 항을 한 번씩 조사하면 됩니다.

---

### 함께 계산하기: `0011`의 점수

앞에서 읽은 `0011`의 두 그룹은

$$
G_0=\{2,3\},
\qquad
G_1=\{0,1\}
$$

입니다. 간선별로 두 끝점의 비트를 비교해 보겠습니다.

| 간선 | 양 끝점의 비트 | 같은 그룹인가? | 간선 점수 |
|---|---|---|---:|
| $(0,1)$ | $1,1$ | 같다 | 0 |
| $(1,2)$ | $1,0$ | 다르다 | 1 |
| $(2,3)$ | $0,0$ | 같다 | 0 |
| $(3,0)$ | $0,1$ | 다르다 | 1 |
| $(0,2)$ | $1,0$ | 다르다 | 1 |

따라서

$$
C(0011)=0+1+0+1+1=3
$$

입니다.

이 계산에서는 정점이 어느 그룹에 속하는지를 먼저 읽고, 간선별로 두 그룹을 비교했습니다. 뒤에서 측정 기록을 평가할 때에도 같은 규칙을 사용합니다.

반대로 모든 정점이 한 그룹에 속하는 `0000`이나 `1111`에서는 간선의 양 끝점이 모두 같은 그룹에 있습니다. 이런 입력은 점수 함수의 기본 동작을 확인하는 데 사용할 수 있습니다.

---

### 가능한 모든 답을 조사해 비교 기준 만들기

정점이 네 개이므로 가능한 비트열은 16개입니다. 각 비트열의 점수를 계산하면 정확한 최대 점수

$$
C_{\max}
=
\max_{z\in\{0,1\}^4}C(z)
$$

를 구할 수 있습니다.

최대 점수를 갖는 비트열을 모두 모으면 최적해 집합이 됩니다.

$$
\mathcal Z_{\mathrm{opt}}
=
\{z:C(z)=C_{\max}\}.
$$

최적해를 찾을 때에는 최대 점수를 처음 달성한 비트열 하나만 기록하지 않습니다. 같은 최대 점수를 갖는 다른 비트열이 있는지도 확인해야 합니다. 이 집합은 나중에 회로가 최적해를 출력할 확률을 계산할 때 사용합니다.

모든 후보를 조사하는 방법을 <strong>전수조사</strong>라고 부릅니다. 이번처럼 작은 문제에서는 16개 후보를 모두 계산할 수 있으므로, 이후 실험 결과를 정확한 기준과 비교할 수 있습니다.

전수조사 표는 여러분이 작성한 `cut_score`를 호출해 만듭니다. 함수가 잘못되면 최대 점수와 최적해 집합도 함께 잘못될 수 있으므로, 먼저 손으로 계산한 사례와 확인 셀을 사용해 점수 함수를 점검하겠습니다.

---

### 균등하게 답을 뽑는 방법을 기준으로 삼기

회로를 최적화한 뒤 결과가 좋아졌는지 판단하려면, 비교할 출발점도 필요합니다. 이번에는 16개 비트열을 같은 확률로 선택하는 <strong>균등 분포</strong>를 사용하겠습니다.

각 비트열의 확률은

$$
P_{\mathrm{unif}}(z)=\frac1{16}
$$

입니다. 따라서 균등 분포의 평균 점수는

$$
\langle C\rangle_{\mathrm{unif}}
=
\frac1{16}\sum_z C(z)
$$

로 계산합니다.

최적해 확률은 최적해 집합에 속하는 비트열들의 확률을 모두 더한 값입니다.

$$
P_{\mathrm{opt,unif}}
=
\sum_{z\in\mathcal Z_{\mathrm{opt}}}\frac1{16}
=
\frac{|\mathcal Z_{\mathrm{opt}}|}{16}.
$$

여기서 $|\mathcal Z_{\mathrm{opt}}|$는 최적 비트열의 개수입니다.

이 두 기준은 뒤에서 사용할 초기 양자 상태와도 연결됩니다. $|+\rangle^{\otimes4}$를 바로 Z 측정하면 각 비트열이 확률 $1/16$로 나오므로, 같은 평균 점수와 최적해 확률을 얻습니다.

아직 회로를 작성하지 않아도, 고전적인 점수표만으로 초기 출력 분포의 성능을 계산할 수 있는 것입니다.

---

### 직접 풀어보기

### (a) 비트열을 분할로 읽고 점수 계산하기

다음 두 비트열을 살펴보세요.

```text
0001
1000
```

각 비트열에서 정점별 그룹을 읽고 표를 완성하세요.

| 비트열 | 그룹 0의 정점들 | 그룹 1의 정점들 | 잘린 간선들 | 점수 |
|---|---|---|---|---:|
| `0001` | … | … | … | … |
| `1000` | … | … | … | … |

계산한 뒤 다음을 설명하세요.

1. 두 문자열에는 모두 1이 하나씩 있습니다. 그런데 두 분할의 점수가 반드시 같아야 할까요? 이번 그래프에서 어떤 간선들이 각 정점에 연결되어 있는지 근거로 설명하세요.
2. 문자열의 왼쪽 끝을 정점 0으로 잘못 읽으면 위 두 입력을 어떻게 해석하게 되나요? 이 두 사례가 비트 순서를 확인하는 데 도움이 되는 이유를 설명하세요.
3. `0001`의 모든 비트를 뒤집은 `1110`은 어떤 두 그룹을 나타내나요? 두 문자열의 점수가 어떻게 비교되는지, 간선의 양 끝점이 같은 그룹인지 다른 그룹인지에 주목하여 설명하세요.

먼저 손으로 계산과 근거를 적으세요. 다음 문항에서 점수 함수를 완성한 뒤, 전수조사 결과와 대조합니다.

---

### 1a 답안

계산·관찰 결과와 근거:

…

### (b) 비트열의 점수를 반환하는 함수 작성하기

다음 함수를 작성하세요.

```python
cut_score(bits)
```

입력 `bits`는 `z3z2z1z0` 순서의 길이 4인 이진 문자열입니다. 입력은 0과 1로만 이루어진 유효한 문자열로 주어집니다.

함수에서 사용할 간선 목록은 다음과 같이 제공됩니다.

```python
EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (0, 2))
```

정점의 비트를 읽을 때에는 제공 도구 `vertex_bit(bits, vertex)`를 사용할 수 있습니다.

함수는 다음 조건을 만족해야 합니다.

- `EDGES`의 각 간선을 한 번씩 조사합니다.
- 양 끝점의 비트가 다르면 그 간선의 점수는 1, 같으면 0입니다.
- 다섯 간선의 점수를 합한 <strong>정수 하나</strong>를 반환합니다.
- 문자열과 정점 번호의 대응을 모든 입력에서 동일하게 유지합니다.

특정 문자열의 점수를 직접 적어 반환하지 말고, 전달받은 문자열과 간선 목록으로 점수를 계산하세요.

먼저 풀이 예제인 `0011`과 (a)에서 계산한 두 입력을 확인합니다. 이어서 확인 셀을 실행하세요. 확인 셀은 가능한 모든 비트열에서 함수의 결과를 독립적인 점수 계산과 비교합니다.

함수가 확인을 통과하면 제공된 전수조사 셀을 실행합니다. 이 셀은 여러분의 함수를 사용해 16개 비트열의 점수표를 만들고, 최대 점수와 그 값을 달성하는 모든 비트열을 보여 줍니다.

---

In [ ]:
def cut_score(bits):
    # TODO BEGIN — 각 간선의 두 끝점이 다른 그룹인지 확인하세요.
    raise NotImplementedError('cut_score')
    # TODO END

In [ ]:
_ = check_cut_score(cut_score)

### 점수표를 확인하는 순서

점수표 관찰 셀은 이어지는 1(c)의 예상 답안 뒤에 있습니다. 최대 점수와 분할을 먼저 예상한 뒤 실행하고, 1(a)의 손계산도 같은 표로 대조하세요.

### 1b 답안

계산·관찰 결과와 근거:

…

### (c) 최대 점수와 균등 분포의 성능 구하기

먼저 그래프의 간선들을 보고, 가능한 최대 점수와 그 점수를 달성할 분할을 예상해 보세요. 예상과 이유를 적은 뒤 (b)의 전수조사 표를 확인합니다.

그 결과를 이용해 다음 표를 완성하세요.

| 항목 | 결과 |
|---|---|
| 최대 점수 $C_{\max}$ | … |
| 최적해 집합 $\mathcal Z_{\mathrm{opt}}$ | … |
| 최적 비트열의 개수 $\vert \mathcal Z_{\mathrm{opt}}\vert $ | … |
| 균등 분포의 평균 점수 $\langle C\rangle_{\mathrm{unif}}$ | … |
| 균등 분포의 최적해 확률 $P_{\mathrm{opt,unif}}$ | … |

표를 완성한 뒤 다음 세 가지를 설명하세요.

<strong>1. 찾은 점수가 실제 최대인 이유</strong>

전수조사 결과 외에도 그래프의 구조로 최대 점수를 설명해 보세요. 특히

$$
(0,1),\qquad(1,2),\qquad(0,2)
$$

의 세 간선을 살펴보세요. 정점 0, 1, 2를 두 그룹에 넣었을 때, 이 세 간선이 모두 잘리도록 할 수 있나요?

이 판단을 전체 다섯 간선의 가능한 점수와 연결하고, 자신이 찾은 분할이 그 한계에 도달하는지 확인하세요.

<strong>2. 균등 분포의 평균 점수를 두 방법으로 확인하기</strong>

첫 번째 방법으로 16개 비트열의 점수를 모두 더한 뒤 16으로 나누세요.

두 번째 방법으로는 간선 하나의 양 끝점에 가능한 비트쌍

$$
(0,0),\ (0,1),\ (1,0),\ (1,1)
$$

을 살펴보세요. 전체 비트열이 균등하게 선택될 때, 한 간선이 잘릴 확률을 구하고 다섯 간선의 평균 기여를 더하세요.

두 방법으로 구한 전체 평균 점수가 일치하는지 확인합니다.

<strong>3. 최적해 확률에 포함할 비트열 판단하기</strong>

최적 비트열에서 모든 비트를 뒤집으면, 바뀐 비트열도 최적해에 속하나요? 두 그룹의 이름과 잘린 간선의 관계를 이용해 설명하세요.

이어서 두 문자열이 같은 분할을 나타내더라도, 측정 결과에서 최적해 확률을 계산할 때에는 왜 <strong>최적해에 해당하는 모든 문자열의 확률을 합해야 하는지</strong> 설명하세요.

### 1c 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 1bc

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_1bc():
    table, maximum, optimal = classical_reference(score_function=cut_score)
    rows = []
    for bits in ("0001", "1000", "1110"):
        groups = [tuple(q for q in range(4) if vertex_bit(bits, q) == bit) for bit in (0, 1)]
        cut_edges = tuple(edge for edge in EDGES if vertex_bit(bits, edge[0]) != vertex_bit(bits, edge[1]))
        rows.append([bits, groups[0], groups[1], cut_edges, table[bits]])
    show_table(["비트열", "그룹 0", "그룹 1", "잘린 간선", "학생 함수의 점수"], rows)
    show_table(["비트열", "점수"], [[bits, score] for bits, score in table.items()])
    uniform_mean = sum(table.values()) / len(table)
    uniform_optimal = len(optimal) / len(table)
    show_table(["최대 점수", "모든 최적 비트열", "균등 평균 점수", "균등 최적해 확률"],
               [[maximum, optimal, uniform_mean, uniform_optimal]])
    return {"score_table": table, "maximum": maximum, "optimal_strings": optimal,
            "uniform_mean": uniform_mean, "uniform_p_opt": uniform_optimal}

if RUN_OBSERVATIONS:
    _ = run_demo("1bc", experiment_1bc, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("1bc", None)
    print("[대기] 1(a,c)의 예상과 계산을 적은 뒤 점수표를 확인하세요.")

## 문제 2. 점수를 양자 회로의 연산으로 어떻게 옮길까요?

앞 문제에서는 비트열 하나의 점수를 계산하고, 가능한 모든 분할을 조사해 비교 기준을 만들었습니다. 이번에는 <strong>그 점수 정보를 양자 회로에 반영하는 방법</strong>을 살펴보겠습니다.

먼저 고전적인 점수 함수를 양자 관측량으로 표현합니다. 이어서 점수에 따라 위상을 붙이는 cost 연산을 만들고, 그 위상 차이가 출력 확률에 영향을 주도록 mixer를 연결합니다.

여러분은 이 과정을 두 함수로 나누어 구현합니다. `cost_layer`는 점수에 따른 위상 연산을 담당하고, `qaoa_p1`은 초기 상태 준비와 cost, mixer를 하나의 회로로 연결합니다.

---

### 간선의 점수를 Z 관측량으로 표현하기

간선 $(i,j)$의 점수는 두 끝점의 비트가 다르면 1, 같으면 0입니다.

$$
c_{ij}(z)
=
\begin{cases}
1,&z_i\ne z_j,\\
0,&z_i=z_j.
\end{cases}
$$

이 조건을 Z 관측량으로 옮겨 보겠습니다. 계산 기저 상태 $|z_i\rangle$에 Z를 적용하면

$$
Z_i|z_i\rangle=(-1)^{z_i}|z_i\rangle
$$

입니다. 따라서 두 큐비트에 $Z_iZ_j$를 적용했을 때의 고윳값은

$$
(-1)^{z_i}(-1)^{z_j}
$$

입니다.

두 비트가 같으면 이 곱은 $+1$, 다르면 $-1$입니다. 이를 간선 점수 0과 1로 바꾸려면

$$
\widehat C_{ij}
=
\frac{I-Z_iZ_j}{2}
$$

를 사용하면 됩니다.

| 두 비트의 관계 | $Z_iZ_j$의 고윳값 | $\widehat C_{ij}$의 고윳값 |
|---|---:|---:|
| 같다 | $+1$ | 0 |
| 다르다 | $-1$ | 1 |

여기서 $I$는 항등 연산자입니다. 아래첨자가 없는 다른 큐비트에는 항등 연산이 작용한다고 생각하면 됩니다.

전체 점수 연산자는 모든 간선의 항을 더한 것입니다.

$$
\widehat C
=
\sum_{(i,j)\in\mathcal E}
\frac{I-Z_iZ_j}{2}.
$$

계산 기저 상태 $|z\rangle$에 대해

$$
\widehat C|z\rangle=C(z)|z\rangle
$$

가 성립합니다. 즉, <strong>고전 점수 함수가 반환하는 값이 점수 연산자의 고윳값</strong>이 됩니다.

일반적인 상태

$$
|\psi\rangle=\sum_z a_z|z\rangle
$$

에서는

$$
\langle\psi|\widehat C|\psi\rangle
=
\sum_z |a_z|^2C(z)
$$

입니다. Z 측정의 출력 확률이 $P(z)=|a_z|^2$이므로, 이 기댓값은 앞에서 정의한 평균 점수와 같습니다.

---

### 점수 관측량에서 위상 연산으로

이제 점수를 계산할 관측량을 얻었습니다. 다음 목표는 이 점수 정보를 상태에 반영하는 것입니다.

점수 연산자 $\widehat C$를 상태에 그대로 적용하면, 각 성분의 진폭에 $C(z)$가 곱해집니다. 일반적으로 상태의 정규화가 유지되지 않으므로, 이 작용을 그대로 유니터리 게이트로 사용할 수는 없습니다.

QAOA에서는 대신 점수에 따라 <strong>절댓값이 1인 위상</strong>을 붙입니다.

$$
U_C(\gamma)
=
e^{-i\gamma\widehat C}.
$$

계산 기저 상태 $|z\rangle$는 $\widehat C$의 고유상태이므로, 이 연산은

$$
U_C(\gamma)|z\rangle
=
e^{-i\gamma C(z)}|z\rangle
$$

로 작용합니다.

여기서 $\gamma$는 실수 각도입니다. 점수가 다른 두 성분에는 서로 다른 위상이 붙을 수 있지만, 각 위상의 절댓값은 1입니다. 따라서 상태의 정규화는 유지됩니다.

중첩 상태에 적용하면

$$
U_C(\gamma)\sum_z a_z|z\rangle
=
\sum_z a_z e^{-i\gamma C(z)}|z\rangle
$$

입니다. 각 성분의 Z 측정 확률은 여전히 $|a_z|^2$이지만, 성분 사이의 상대위상은 달라질 수 있습니다.

이것이 이번 cost 연산의 역할입니다. <strong>점수 정보를 먼저 상대위상에 담고, 뒤의 mixer가 그 차이를 출력 진폭에 반영하도록 합니다.</strong>

---

### 전체 cost를 간선별 연산으로 나누기

전체 점수는 간선 점수의 합이므로

$$
C(z)=\sum_{(i,j)\in\mathcal E}c_{ij}(z)
$$

입니다. 따라서 비트열 하나에 붙일 위상은

$$
e^{-i\gamma C(z)}
=
\prod_{(i,j)\in\mathcal E}
e^{-i\gamma c_{ij}(z)}
$$

로 나눌 수 있습니다.

연산자로도 같은 분해가 가능합니다. 각 간선의 $\widehat C_{ij}$는 I와 Z로만 이루어져 있으므로 서로 교환합니다. 간선들이 같은 정점을 공유하더라도, 그 큐비트에서 작용하는 연산이 모두 Z이기 때문입니다.

따라서

$$
U_C(\gamma)
=
\prod_{(i,j)\in\mathcal E}
e^{-i\gamma\widehat C_{ij}}
$$

로 구현할 수 있습니다.

회로에서는 다섯 간선에 해당하는 연산을 차례대로 적용하면 됩니다. 이 cost 연산들 사이의 순서는 바꾸어도 같은 결과를 줍니다. 다만 뒤에 적용할 mixer와의 순서까지 자유롭게 바꿀 수 있다는 뜻은 아닙니다.

---

### 한 간선의 위상을 $R_{ZZ}$로 구현하기

한 간선 $(i,j)$에 필요한 연산은

$$
e^{-i\gamma\widehat C_{ij}}
=
e^{-i\gamma(I-Z_iZ_j)/2}
$$

입니다.

두 끝점의 비트가 같으면 간선 점수가 0이므로 위상 1이 붙고, 다르면 점수가 1이므로 $e^{-i\gamma}$가 붙습니다.

| 두 끝점의 비트 | 간선 점수 | 필요한 위상 |
|---|---:|---|
| 00 | 0 | $1$ |
| 01 | 1 | $e^{-i\gamma}$ |
| 10 | 1 | $e^{-i\gamma}$ |
| 11 | 0 | $1$ |

이 작용을 구현할 때 사용할 게이트는

$$
R_{ZZ}(\lambda)
=
e^{-i\lambda Z_iZ_j/2}
$$

입니다. $Z_iZ_j$의 고윳값이 같은 비트에서 $+1$, 다른 비트에서 $-1$이므로 이 게이트의 작용은 다음과 같습니다.

$$
R_{ZZ}(\lambda)
=
\begin{pmatrix}
e^{-i\lambda/2}&0&0&0\\
0&e^{i\lambda/2}&0&0\\
0&0&e^{i\lambda/2}&0\\
0&0&0&e^{-i\lambda/2}
\end{pmatrix}.
$$

여러분은 두 표현을 비교하여 필요한 회전각 $\lambda$를 정해야 합니다.

이때 모든 성분에 같은 위상이 곱해지는 차이는 허용할 수 있습니다. 따라서 네 대각 성분이 문자 그대로 같은지만 보기보다, <strong>같은 비트 성분과 다른 비트 성분 사이에 필요한 상대위상이 만들어지는지</strong> 확인하세요.

한 간선에서 생기는 공통 위상은 네 큐비트 전체의 상태에도 공통으로 곱해집니다. 다섯 간선의 공통 위상을 모두 곱한 것 역시 전체 상태의 전역위상이므로, 물리적 상태 비교에서는 허용합니다.

---

### Mixer는 진폭들을 결합합니다

Cost 연산만으로는 Z 측정 확률이 달라지지 않습니다. 다음으로 각 큐비트의 0과 1 성분을 결합하는 연산을 적용하겠습니다.

이번 mixer는

$$
\widehat B=\sum_{i=0}^{3}X_i,
\qquad
U_B(\beta)=e^{-i\beta\widehat B}
$$

로 정의합니다.

서로 다른 큐비트에 작용하는 X 연산들은 교환하므로

$$
U_B(\beta)
=
\prod_{i=0}^{3}e^{-i\beta X_i}
$$

로 구현할 수 있습니다.

사용할 한 큐비트 회전 게이트의 정의는

$$
R_x(\lambda)
=
e^{-i\lambda X/2}
=
\begin{pmatrix}
\cos(\lambda/2)&-i\sin(\lambda/2)\\
-i\sin(\lambda/2)&\cos(\lambda/2)
\end{pmatrix}
$$

입니다. 이 정의와 $e^{-i\beta X_i}$를 비교하여 각 큐비트에 넣을 회전각을 정합니다.

이 행렬을 보면 출력의 0 성분에는 입력의 0 성분과 1 성분이 모두 기여합니다. 출력의 1 성분도 마찬가지입니다. 즉, mixer는 비트열의 확률을 단순히 평균내는 것이 아니라 <strong>복소 진폭들을 유니터리 연산으로 결합</strong>합니다.

간단한 한 큐비트 상태를 예로 들어 보겠습니다.

$$
|\chi_\phi\rangle
=
\frac{|0\rangle+e^{i\phi}|1\rangle}{\sqrt2}.
$$

이 상태의 Z 측정 확률은 $\phi$와 관계없이 0과 1이 각각 $1/2$입니다. 하지만 $R_x(\lambda)$를 적용한 뒤의 0 성분 진폭은

$$
\frac{
\cos(\lambda/2)
-i e^{i\phi}\sin(\lambda/2)
}{\sqrt2}
$$

가 됩니다.

서로 다른 두 성분에서 온 진폭이 한 출력 성분에 더해졌고, 그 합에는 상대위상 $\phi$가 포함됩니다. 따라서 합의 절댓값 제곱인 측정 확률이 $\phi$에 따라 달라질 수 있습니다.

네 큐비트의 QAOA에서도 같은 원리가 작용합니다. Cost가 만든 상대위상이 mixer를 거치며 출력 확률에 영향을 줄 수 있고, 그 결과 평균 점수도 달라질 수 있습니다. 어떤 각도가 좋은 결과를 주는지는 실제 계산으로 확인해야 합니다.

---

### 초기 상태부터 한 층을 조립하기

이번 $p=1$ 회로의 상태는

$$
|\psi(\gamma,\beta)\rangle
=
U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$

입니다.

회로에서는 다음 순서로 준비합니다.

| 순서 | 수행할 일 |
|---|---|
| 1 | $\vert 0000\rangle$에서 시작합니다. |
| 2 | 각 큐비트에 H를 적용하여 $\vert +\rangle^{\otimes4}$를 준비합니다. |
| 3 | 다섯 간선의 cost 연산을 적용합니다. |
| 4 | 네 큐비트에 mixer 연산을 적용합니다. |

수식에서는 오른쪽 연산부터 상태에 작용하므로, $U_BU_C$는 <strong>cost 다음 mixer</strong>를 뜻합니다.

회로를 작성할 때 사용할 문법은 다음과 같습니다.

| 문법 | 역할 |
|---|---|
| `circuit.h(q)` | q에 H 적용 |
| `circuit.rzz(angle, i, j)` | i와 j에 지정한 각도의 $R_{ZZ}$ 적용 |
| `circuit.rx(angle, q)` | q에 지정한 각도의 $R_x$ 적용 |
| `circuit.compose(layer, inplace=True)` | 같은 큐비트 순서를 사용하는 `layer`를 현재 회로 뒤에 연결 |

회로를 붙이는 문법을 한 큐비트 예로 확인하면 다음과 같습니다.

```python
layer = QuantumCircuit(1)
layer.x(0)

combined = QuantumCircuit(1)
combined.compose(layer, inplace=True)
```

이제 `combined`에는 `layer`의 X 게이트가 들어 있습니다. 이번 문제에서도 같은 방식으로, 여러분이 작성한 cost 회로를 전체 QAOA 회로에 연결할 수 있습니다.

---

### 직접 풀어보기

### (a) 점수와 양자 연산자의 작용 비교하기

문제 1에서 계산한 다음 두 비트열을 다시 사용하겠습니다.

```text
0001
1000
```

각 비트열의 점수를 사용하여 표를 완성하세요.

| 비트열 $z$ | 고전 점수 $C(z)$ | $\widehat C\vert z\rangle$ | $U_C(\gamma)\vert z\rangle$ |
|---|---:|---|---|
| `0001` | … | … | … |
| `1000` | … | … | … |

이어서 두 성분의 중첩 상태

$$
|\eta\rangle
=
\frac{|0001\rangle+|1000\rangle}{\sqrt2}
$$

를 생각해 보세요.

1. $\widehat C|\eta\rangle$와 $U_C(\gamma)|\eta\rangle$를 각각 구하세요.
2. 두 결과에서 각 성분의 계수가 어떻게 달라졌는지 비교하세요. 어느 연산이 상태의 정규화를 유지하나요?
3. Cost 연산 뒤에도 두 비트열의 Z 측정 확률이 각각 $1/2$인지 확인하세요.
4. 두 성분에 붙은 위상이 항상 전역위상 하나로 묶이는지, 아니면 $\gamma$에 따라 상대위상이 달라질 수 있는지 설명하세요.

마지막으로 제공된 연산자 계산 도구를 사용해, 16개 계산 기저 상태에서 $\widehat C$의 고윳값과 자신이 작성한 `cut_score`의 결과가 일치하는지 확인합니다.

전체 $16\times16$ 행렬을 손으로 전개할 필요는 없습니다. 각 계산 기저가 고유상태이고, 그 고윳값이 분할의 점수라는 대응을 확인하세요.

---

### 2a 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 2a

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_2a():
    diagonal = np.diag(cost_operator().to_matrix()).real
    rows = [[bits, cut_score(bits), diagonal[int(bits, 2)]] for bits in BITSTRINGS]
    show_table(["비트열", "학생 고전 점수", "점수 연산자 고윳값"], rows)
    gamma = 0.37
    initial = np.zeros(16, dtype=complex)
    initial[int("0001", 2)] = 1 / np.sqrt(2)
    initial[int("1000", 2)] = 1 / np.sqrt(2)
    observed = diagonal * initial
    phased = np.einsum("ij,j->i", definition_cost_matrix(gamma), initial)
    show_table(["비트열", "원래 진폭", "C 적용 진폭", "U_C 적용 진폭"],
               [[bits, initial[int(bits, 2)], observed[int(bits, 2)], phased[int(bits, 2)]] for bits in ("0001", "1000")])
    show_table(["비교", "벡터 norm 제곱"], [["원래", float(np.vdot(initial, initial).real)],
                                                ["C 적용", float(np.vdot(observed, observed).real)],
                                                ["U_C 적용", float(np.vdot(phased, phased).real)]])
    print("위상 관찰에는 gamma=", gamma, "를 사용했습니다. 일반 gamma의 상태식은 별도로 작성하세요.")
    return {"diagonal": diagonal, "gamma": gamma, "initial": initial,
            "cost_applied": observed, "unitary_applied": phased}

if RUN_OBSERVATIONS:
    _ = run_demo("2a", experiment_2a, (CHECK_LABELS[0],))
else:
    LAB_RESULTS.pop("2a", None)
    print("[대기] 2(a)의 점수와 위상 작용을 먼저 예상하세요.")

### (b) 간선별 cost 연산 구현하기

다음 함수를 작성하세요.

```python
cost_layer(gamma, cost_scale=1.0)
```

`cost_scale`은 전체 점수에 곱하는 양의 실수 배율입니다. 현재 실험에서는 기본값 1.0을 사용합니다. 마지막 적용에서 점수 단위를 바꾸더라도 같은 도구를 사용할 수 있도록 이 매개변수를 두겠습니다.

배율을 $s$라고 쓰면 함수가 구현할 연산은

$$
U_C(\gamma;s)
=
e^{-i\gamma s\widehat C}
$$

입니다. 모든 간선의 점수에 같은 배율이 적용되며, 그래프의 연결은 그대로 유지됩니다.

먼저 한 간선의 연산

$$
e^{-i\gamma s(I-Z_iZ_j)/2}
$$

을 $R_{ZZ}(\lambda)$와 비교하여 다음을 구하세요.

- 필요한 회전각 $\lambda$
- 두 표현 사이에 남는 전역위상
- 간선별 연산의 순서를 바꾸어도 되는 이유

그 결과를 이용해 함수를 작성합니다.

함수의 조건은 다음과 같습니다.

- 반환값은 큐비트 네 개, 고전 비트가 없는 `QuantumCircuit(4)`입니다.
- `EDGES`의 다섯 간선에 해당하는 cost 연산을 모두 포함합니다.
- $R_{ZZ}$를 사용하거나, 이를 동등한 CX–$R_z$–CX로 분해할 수 있습니다.
- 초기 균등 중첩을 만드는 H와 mixer는 전체 회로에서 담당합니다. 이 함수는 cost 연산만 반환합니다.
- 측정, reset, initialize를 넣지 않습니다.
- 일반적인 실수 $\gamma$와 양의 `cost_scale`에서 작동해야 합니다.
- 전체 연산에 공통으로 곱해지는 전역위상은 허용합니다.

Cost layer는 이미 준비된 상태에 적용할 연산입니다. 따라서 $|0000\rangle$ 한 상태에서만 결과가 맞는 것으로는 충분하지 않습니다. 확인 셀은 여러 각도와 배율에서 <strong>전체 cost 연산이 정의와 일치하는지</strong> 검사합니다.

---

In [ ]:
def cost_layer(gamma, cost_scale=1.0):
    circuit = QuantumCircuit(4)
    # TODO BEGIN — 점수와 배율에 따른 cost 연산을 구현하세요.
    raise NotImplementedError('cost_layer')
    # TODO END
    return circuit

In [ ]:
_ = check_cost_layer(cost_layer)

### 2b 답안

계산·관찰 결과와 근거:

…

### (c) $p=1$ QAOA 회로 작성하기

다음 함수를 작성하세요.

```python
qaoa_p1(gamma, beta, cost_scale=1.0)
```

함수는 $|0000\rangle$에서 시작하여

$$
|\psi(\gamma,\beta;s)\rangle
=
U_B(\beta)U_C(\gamma;s)|+\rangle^{\otimes4}
$$

를 준비하는 회로를 반환해야 합니다. 기본 실험에서는 $s=1$입니다.

먼저

$$
e^{-i\beta X}
\quad\text{와}\quad
R_x(\lambda)=e^{-i\lambda X/2}
$$

를 비교하여 mixer에 사용할 회전각 $\lambda$를 구하세요.

그다음 다음 세 부분을 순서대로 연결합니다.

1. 네 큐비트의 균등 중첩 준비
2. 자신이 작성한 `cost_layer`가 반환하는 회로
3. 네 큐비트의 mixer

함수의 조건은 다음과 같습니다.

- 반환값은 큐비트 네 개, 고전 비트가 없는 `QuantumCircuit(4)`입니다.
- 전달받은 `gamma`, `beta`, `cost_scale`을 사용합니다.
- Cost 부분은 자신이 작성한 `cost_layer`를 호출하여 연결합니다.
- H, $R_{ZZ}$, $R_x$를 사용하며, cost의 동등한 CX–$R_z$–CX 분해도 허용합니다.
- 측정, reset, initialize와 추가 큐비트는 사용하지 않습니다.
- 특정 최적해 비트열이나 특정 각도에만 맞는 회로가 아니라, 일반적인 입력 각도에서 정의된 상태를 준비해야 합니다.

회로를 작성한 뒤 다음을 설명하세요.

- H는 어떤 초기 분포를 준비하기 위해 사용했나요?
- Cost와 mixer에는 각각 어떤 각도를 전달했으며, 그 계수와 부호를 어떻게 정했나요?
- 수식의 $U_BU_C$ 순서를 코드에서는 어떤 순서로 구현했나요?
- `cost_scale`이 달라질 때 어떤 부분에 그 값이 반영되나요?

확인 셀은 여러 입력에서 준비된 공동 상태를 정의에 따른 상태와 비교합니다. 전역위상과 동등한 게이트 구현은 허용하지만, <strong>일부 측정 확률이나 최종 점수만 우연히 같은 회로를 같은 구현으로 판단하지는 않습니다.</strong>

In [ ]:
def qaoa_p1(gamma, beta, cost_scale=1.0):
    circuit = QuantumCircuit(4)
    # TODO BEGIN — 초기 상태, 자신의 cost_layer, mixer를 연결하세요.
    raise NotImplementedError('qaoa_p1')
    # TODO END
    return circuit

In [ ]:
_ = check_qaoa_p1(qaoa_p1)

### 2c 답안

계산·관찰 결과와 근거:

…

## 문제 3. 두 연산이 실제로 출력 분포를 바꿀까요?

앞 문제에서는 점수에 따라 위상을 붙이는 cost 연산과, 그 위상 차이가 출력 진폭에 영향을 주도록 하는 mixer를 연결했습니다. 이제 <strong>회로의 각도를 바꾸어 평균 점수를 높이고, 두 연산이 어떤 역할을 했는지 확인</strong>하겠습니다.

이번 문제에서는 그래프와 간선의 점수를 유지합니다. `cost_scale`은 1.0으로 고정하고, cost 각도 $\gamma$와 mixer 각도 $\beta$만 조절합니다.

먼저 고전 최적화기에 전달할 값을 확인합니다. 이어서 최적화를 실행하고, 같은 각도에서 mixer를 빼거나 연산 순서를 바꾼 회로를 비교하겠습니다.

---

### 이번 목표는 평균 점수를 높이는 것입니다

각도 $(\gamma,\beta)$에서 회로가 준비한 상태를

$$
|\psi(\gamma,\beta)\rangle
$$

라고 하겠습니다. 이 상태를 Z 측정하여 비트열 $z$를 얻을 확률이 $P_{\gamma,\beta}(z)$라면, 평균 점수는

$$
\mu(\gamma,\beta)
=
\sum_z P_{\gamma,\beta}(z)C(z)
$$

입니다.

앞 문제에서 만든 점수 연산자를 사용하면 같은 값을

$$
\mu(\gamma,\beta)
=
\langle\psi(\gamma,\beta)|
\widehat C
|\psi(\gamma,\beta)\rangle
$$

로 계산할 수 있습니다.

여기서 $\mu$는 <strong>각도에 따라 달라지는 평균 cut</strong>을 나타냅니다. 높은 점수의 비트열에 더 많은 확률이 배정되면 평균 점수가 높아질 수 있습니다.

각 비트열의 점수는 0 이상이고 $C_{\max}$ 이하이므로, 정규화된 출력 분포의 평균도

$$
0\le\mu(\gamma,\beta)\le C_{\max}
$$

를 만족합니다. 따라서 문제 1에서 구한 최대 점수는 평균 점수를 해석할 때에도 비교 기준이 됩니다.

---

### 최대화할 값을 최소화기에 전달하기

이번에 제공되는 고전 최적화기는 입력받은 목적함수의 값을 <strong>작게 만드는 방향</strong>으로 탐색합니다. 그러나 우리가 높이려는 값은 평균 점수 $\mu$입니다.

이 둘을 연결하기 위해 목적함수를

$$
f(\gamma,\beta)=-\mu(\gamma,\beta)
$$

로 정합니다.

예를 들어 어떤 두 후보의 평균 점수가 각각 2.4와 2.9라면, 목적함수 값은 각각 $-2.4$와 $-2.9$입니다. 최소화기는 더 작은 값인 $-2.9$를 선호하며, 이는 더 높은 평균 점수 2.9에 해당합니다.

이때 세 가지 부호를 구분해야 합니다.

| 표현 | 역할 |
|---|---|
| $\widehat C$ | 분할의 점수를 나타내며, 높은 값이 좋은 답에 해당합니다. |
| $U_C(\gamma)=e^{-i\gamma\widehat C}$ | 이번 실습에서 정한 cost 회로의 정의입니다. |
| $f(\gamma,\beta)=-\langle\widehat C\rangle$ | 평균 점수를 최대화하기 위해 최소화기에 전달하는 값입니다. |

목적함수에 붙인 음수는 <strong>고전 최적화기가 결과를 비교하는 방향</strong>을 정합니다. 회로의 cost 연산과 회전 게이트의 각도는 문제 2에서 정한 정의를 따릅니다. 목적함수가 음수를 반환한다는 이유로 cost 회로의 부호를 따로 뒤집지 않습니다.

---

### 제공된 목적함수가 학생 회로를 사용하는 과정

이번에는 목적함수와 고전 최적화기 연결 코드를 제공합니다. 여러분은 그 코드가 어떤 값을 계산하는지 확인하고, 앞에서 작성한 회로를 연결해 실행합니다.

평균 점수를 계산하는 제공 도구는 다음과 같습니다.

```python
mean_cut_from_circuit(circuit, cost_scale=1.0)
```

이 도구는 전달된 회로가 $|0000\rangle$에서 준비한 상태를 계산하고, 점수 연산자의 정확한 기댓값을 반환합니다. 현재 문제에서는 `cost_scale=1.0`을 사용합니다.

최적화기에 전달할 목적함수는 다음과 같습니다.

```python
def qaoa_objective(parameters):
    gamma, beta = parameters
    circuit = qaoa_p1(gamma, beta, cost_scale=1.0)
    mean_cut = mean_cut_from_circuit(circuit, cost_scale=1.0)
    return -mean_cut
```

`qaoa_p1`은 여러분이 작성한 함수입니다. 그 안에서는 여러분의 `cost_layer`가 사용됩니다. 따라서 최적화기가 각도를 제안할 때마다 실제로 작성한 회로의 출력이 평가됩니다.

이 목적함수는 새로운 TODO가 아닙니다. 이번 문항에서는 입력 각도, 준비되는 상태, 평균 점수, 반환값의 관계를 확인하겠습니다.

앞부분의 최적화에는 정확한 기댓값을 사용합니다. 유한 shots로 측정한 기록의 변동은 다음 문제에서 다루겠습니다.

---

### 제공 목적함수 실행

아래 셀도 실행하세요. 위에서 설명한 목적함수를 정의하며, 새 함수 TODO는 아닙니다.

In [ ]:
def qaoa_objective(parameters):
    gamma, beta = parameters
    circuit = qaoa_p1(gamma, beta, cost_scale=1.0)
    mean_cut = mean_cut_from_circuit(circuit, cost_scale=1.0)
    return -mean_cut

### 각도를 갱신할 때마다 새로운 상태를 평가합니다

제공된 최적화 실험은 초기값

$$
(\gamma,\beta)=(0.1,0.1)
$$

에서 시작합니다. 각도 단위는 라디안입니다.

최적화기는 여러 각도를 목적함수에 전달하고, 반환된 값을 비교해 다음 후보를 선택합니다. 한 각도를 평가할 때마다 $|0000\rangle$에서 상태 준비를 다시 시작합니다. 앞의 후보에서 만든 상태에 다음 후보의 게이트를 계속 이어 붙이는 절차가 아닙니다.

실행 기록에는 다음 정보가 남습니다.

| 기록 | 의미 |
|---|---|
| 입력 각도 | 해당 평가에서 사용한 $\gamma,\beta$ |
| 평균 점수 | 그 회로 상태의 정확한 $\langle\widehat C\rangle$ |
| 목적함수 값 | 최적화기에 반환한 $-\langle\widehat C\rangle$ |
| 평가 횟수 | 목적함수를 호출한 횟수 |
| 종료 정보 | 최적화기가 탐색을 마친 상태와 이유 |

최적화기는 주변 후보를 비교하기 위해 더 낮은 평균 점수의 각도도 시험할 수 있습니다. 따라서 평가 기록의 평균 점수가 매번 증가할 필요는 없습니다. 또한 한 번의 최적화 단계에서 목적함수를 여러 번 호출할 수 있으므로, 평가 횟수와 반복 횟수도 구분합니다.

최적화가 끝나면 반환된 각도로 회로를 다시 준비하여 평균 점수를 확인합니다. 그 값은 <strong>이번 실행에서 찾은 결과</strong>입니다. 가능한 모든 각도를 조사한 최댓값의 증명으로 해석하지 않습니다.

---

### 연산을 바꾸어 각 부분의 역할 확인하기

평균 점수가 달라졌다는 사실을 확인한 뒤에는, 어떤 연산이 그 변화를 만들었는지 살펴보겠습니다.

정상 회로는

$$
U_B(\beta)U_C(\gamma)|+\rangle^{\otimes4}
$$

입니다. 이번에는 각도를 고정하고 다음 경우를 비교합니다.

| 비교 대상 | 적용하는 연산 |
|---|---|
| 초기 상태 | $\lvert+\rangle^{\otimes4}$를 바로 측정 |
| 정상 회로 | Cost 다음 mixer |
| Cost만 적용 | 초기 상태에 cost만 적용 |
| 순서를 바꾼 회로 | Mixer 다음 cost |

두 연산의 순서를 바꾸면 일반적으로 다른 상태가 만들어집니다. 하지만 결과를 해석할 때에는 “순서가 다르므로 반드시 모든 측정값이 달라진다”고 판단해서도 안 됩니다. <strong>처음 작용하는 연산이 주어진 초기 상태에 어떤 영향을 주는지</strong>부터 확인해야 합니다.

예를 들어 mixer를 먼저 적용하는 경우에는

$$
X|+\rangle=|+\rangle
$$

라는 관계를 사용할 수 있습니다. 문제 2에서 고유상태에 지수형 연산을 적용한 방법을 떠올려, 이 관계가 mixer의 작용에 무엇을 알려 주는지 살펴보세요.

비교 회로들은 제공됩니다. 새로운 답안 함수를 만들 필요는 없으며, 같은 그래프와 같은 각도에서 연산의 유무와 순서만 바꾸어 결과를 해석합니다.

---

### 같은 Z 분포와 같은 양자 상태는 구분해야 합니다

Cost 연산은 계산 기저 성분에 위상을 붙입니다. 따라서 Z 측정 분포가 그대로이더라도, 상태의 상대위상은 달라질 수 있습니다.

이번 비교에서는 두 가지를 따로 확인하겠습니다.

첫째는 초기 균등 분포와 출력 분포의 차이입니다. 제공 도구는

$$
D_Z
=
\max_z\left|P(z)-\frac1{16}\right|
$$

를 계산합니다. 이는 16개 비트열 가운데 초기 확률 $1/16$에서 가장 크게 달라진 정도입니다. $D_Z=0$이면 Z 측정 분포가 초기 균등 분포와 같습니다.

둘째는 초기 양자 상태와의 fidelity입니다.

$$
F_{\mathrm{initial}}
=
\left|
\langle+|^{\otimes4}
|\psi_{\mathrm{out}}\rangle
\right|^2.
$$

$F_{\mathrm{initial}}=1$이면 출력은 초기 상태와 전역위상을 제외하고 같습니다. 이 값은 <strong>초기 상태가 그대로 유지되었는지 확인하기 위한 비교</strong>입니다. 최적 비트열을 얻을 확률이나 최적화의 성능을 나타내는 값은 아닙니다.

따라서 $D_Z$와 $F_{\mathrm{initial}}$을 함께 보면, 측정 분포가 같은 경우에도 상태의 상대위상이 달라졌는지 구분할 수 있습니다. 계산 결과에서는 작은 수치 오차를 고려하여 제공된 허용 오차 안에서 비교합니다.

---

### 직접 풀어보기

### (a) 목적함수의 부호와 평가 방향 확인하기

다음 두 후보 각도를 사용하겠습니다.

$$
(\gamma_P,\beta_P)=(0.1,0.1),
\qquad
(\gamma_Q,\beta_Q)=(0.3,0.2).
$$

제공 도구로 두 회로의 정확한 평균 점수를 계산하고 표를 완성하세요.

| 후보 | $(\gamma,\beta)$ | 평균 점수 $\mu$ | 목적함수 값 $f$ |
|---|---|---:|---:|
| P | $(0.1,0.1)$ | … | … |
| Q | $(0.3,0.2)$ | … | … |

그 결과를 이용해 다음을 설명하세요.

1. 두 후보 중 평균 점수가 더 높은 것은 어느 쪽인가요? 최소화기가 더 작은 목적함수 값으로 선택하는 후보와 일치하나요?
2. 목적함수가 음수 부호 없이 평균 점수를 그대로 반환한다면, 최소화기는 어떤 방향을 선호하게 되나요?
3. 평균 점수를 최대화하기 위해 목적함수에 음수를 붙이는 것과, cost 회로의 정의를 바꾸는 것은 왜 다른 일인가요?

마지막으로 각 후보의 평균 점수가

$$
0\le\mu\le C_{\max}
$$

범위에 있는지 확인하세요. 목적함수 값의 범위도 같은 기준에서 구해 적습니다.

---

### 3a 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 3a

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_3a():
    rows, results = [], []
    for label, parameters in (("P", (0.1, 0.1)), ("Q", (0.3, 0.2))):
        circuit = qaoa_p1(*parameters, cost_scale=1.0)
        mean = mean_cut_from_circuit(circuit, cost_scale=1.0)
        objective = qaoa_objective(parameters)
        rows.append([label, parameters, mean, objective])
        results.append({"label": label, "parameters": parameters, "mean_cut": mean, "objective_value": objective})
    show_table(["후보", "gamma,beta", "정확한 평균 점수", "목적함수 값"], rows)
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("3a", experiment_3a, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3a", None)
    print("[대기] 3(a)의 최대화·최소화 방향을 확인한 뒤 실행하세요.")

### (b) 각도를 최적화하고 출발점과 비교하기

앞의 회로 확인 셀을 통과한 뒤, 제공된 최적화 실험을 실행하세요. 초기값은 $(0.1,0.1)$입니다.

다음 세 경우를 같은 평균 점수 지표로 비교합니다.

- 문제 1에서 계산한 균등 분포
- 초기 각도 $(0.1,0.1)$의 정상 QAOA 회로
- 최적화기가 반환한 각도의 정상 QAOA 회로

| 비교 대상 | 사용한 각도 | 정확한 평균 점수 | $C_{\max}-\langle C\rangle$ |
|---|---|---:|---:|
| 균등 분포 | 해당 없음 | … | … |
| 초기 각도의 회로 | $(0.1,0.1)$ | … | … |
| 최적화 후 회로 | … | … | … |

최적화기의 반환값과 실행 정보도 기록하세요.

| 항목 | 결과 |
|---|---|
| 반환된 $\gamma,\beta$ | … |
| 반환된 목적함수 값 | … |
| 반환된 각도에서 다시 계산한 평균 점수 | … |
| 목적함수 평가 횟수 | … |
| 최적화 반복 횟수 | … |
| 종료 메시지 | … |

이어서 다음을 설명하세요.

- 초기 각도와 비교했을 때 평균 점수가 개선되었나요? 균등 분포와 비교하면 어떤가요?
- 반환된 목적함수 값과 다시 계산한 평균 점수가 부호 관계에 맞게 일치하나요?
- 평균 점수가 $C_{\max}$보다 낮다면, 이번 출력이 최적해에만 확률을 주는 분포라고 할 수 있나요?
- 이번 최적화 결과만으로, 이 $p=1$ 회로의 모든 각도 가운데 가장 좋은 값을 찾았다고 결론 내릴 수 있나요?

평가 기록 중 일부 점수가 이전보다 낮아졌다면 그 기록도 살펴보세요. 최적화 과정에서 여러 후보를 시험한다는 점과, 최종적으로 반환한 각도의 성능을 구분해 설명합니다.

---

### 3b 답안

계산·관찰 결과와 근거:

…

### 관찰 실험 3b

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_3b():
    report = optimize_qaoa(qaoa_objective, initial=(0.1, 0.1))
    circuit = qaoa_p1(*report["parameters"], cost_scale=1.0)
    report["recalculated_mean"] = mean_cut_from_circuit(circuit, cost_scale=1.0)
    report["recalculated_objective"] = qaoa_objective(report["parameters"])
    assert np.isclose(report["recalculated_mean"], report["mean_cut"], atol=ATOL, rtol=0), "반환된 각도에서의 평균을 다시 확인하세요."
    table, maximum, optimal = classical_reference(score_function=cut_score)
    report["maximum"] = maximum
    report["uniform_mean"] = sum(table.values()) / len(table)
    show_table(["비교 대상", "각도", "평균 점수", "Cmax - 평균"],
               [["균등 분포", "해당 없음", report["uniform_mean"], maximum-report["uniform_mean"]],
                ["초기 회로", report["initial"], report["initial_mean"], maximum-report["initial_mean"]],
                ["최적화 반환 회로", report["parameters"], report["recalculated_mean"], maximum-report["recalculated_mean"]]])
    show_table(["반환 목적함수 값", "평가 횟수", "반복 횟수", "정상 종료"],
               [[report["objective_value"], report["nfev"], report["nit"], report["success"]]])
    print("종료 메시지:", report["message"])
    print("처음·마지막 평가 기록(전체 기록은 LAB_RESULTS['3b']['history']):")
    show_table(["평가 순서", "gamma,beta", "평균 점수", "목적함수 값"],
               [[index+1, report["history"][index]["parameters"], report["history"][index]["mean_cut"],
                 report["history"][index]["objective_value"]] for index in (0, len(report["history"])-1)])
    print("이번 실행에서 찾은 결과입니다. 모든 각도의 전역 최댓값을 증명한 것은 아닙니다.")
    return report

if RUN_OBSERVATIONS:
    _ = run_demo("3b", experiment_3b, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3b", None)
    print("[대기] 3(b): 앞의 회로 검사 후 관찰 실행을 켜세요.")

### (c) Mixer를 빼거나 연산 순서를 바꾸면 무엇이 달라질까요?

(b)에서 얻은 각도를 $(\gamma_*,\beta_*)$라고 하겠습니다. 두 값을 고정하고 다음 네 경우를 비교하세요.

| 비교 대상 | 출력 상태의 정의 |
|---|---|
| 초기 상태 | $\lvert+\rangle^{\otimes4}$ |
| 정상 회로 | $U_B(\beta_*)U_C(\gamma_*)\lvert+\rangle^{\otimes4}$ |
| Cost만 적용 | $U_C(\gamma_*)\lvert+\rangle^{\otimes4}$ |
| 순서를 바꾼 회로 | $U_C(\gamma_*)U_B(\beta_*)\lvert+\rangle^{\otimes4}$ |

실행 전에 각 경우의 Z 측정 분포가 균등하게 남을지, 평균 점수가 어떻게 비교될지 예상하세요. 그런 다음 제공된 비교 셀을 실행하여 표를 완성합니다.

| 비교 대상 | 실행 전 예상과 이유 | $D_Z$ | 정확한 평균 점수 | $F_{\mathrm{initial}}$ |
|---|---|---:|---:|---:|
| 초기 상태 | … | … | … | … |
| 정상 회로 | … | … | … | … |
| Cost만 적용 | … | … | … | … |
| 순서를 바꾼 회로 | … | … | … | … |

다음 내용을 설명하세요.

<strong>1. Cost만 적용한 경우</strong>

초기 상태의 각 비트열 진폭 $1/4$에 어떤 위상이 붙는지 쓰세요. 그 진폭의 절댓값 제곱으로 Z 측정 분포를 설명하고, 같은 분포가 나온다는 사실이 초기 양자 상태와 같다는 뜻인지도 판단하세요.

이 회로가 정상 QAOA 회로에서 $\beta$만 0으로 바꾼 경우와 같은지 확인합니다. 이때 $\gamma$는 $\gamma_*$로 유지합니다.

<strong>2. Mixer를 먼저 적용한 경우</strong>

각 큐비트가 $|+\rangle$인 초기 상태에 mixer가 어떤 작용을 하는지 구하세요. 다음 두 관계를 사용해도 됩니다.

$$
X|+\rangle=|+\rangle,
\qquad
U_B(\beta)=\prod_{i=0}^{3}e^{-i\beta X_i}.
$$

Mixer를 거친 뒤 cost까지 적용한 상태를, cost만 적용한 상태와 비교하세요. 관찰한 분포와 평균 점수가 왜 그렇게 나왔는지 설명합니다.

<strong>3. 두 연산의 역할 정리하기</strong>

정상 회로에서 cost와 mixer가 각각 무엇을 했는지 설명하세요. “게이트가 많아서 좋아졌다”는 설명 대신, <strong>점수에 따른 상대위상과 그 뒤의 진폭 결합</strong>을 연결합니다.

또한 출력 분포가 초기 상태와 같았던 경우가 있다면, $F_{\mathrm{initial}}$과 상태식을 함께 확인하여 “아무 연산도 하지 않은 것과 같다”는 주장이 적절한지 판단하세요.

### 3c 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 3c

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_3c():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b)의 최적화를 실행하세요.")
    gamma, beta = LAB_RESULTS["3b"]["parameters"]
    cost = circuit_unitary(cost_layer(gamma, cost_scale=1.0), "cost")
    mixer = expm(-1j * beta * MIXER_OPERATOR.to_matrix())
    after_cost = Statevector(np.einsum("ij,j->i", cost, INITIAL_STATE.data))
    after_mixer = np.einsum("ij,j->i", mixer, INITIAL_STATE.data)
    reversed_state = Statevector(np.einsum("ij,j->i", cost, after_mixer))
    states = [("초기 상태", INITIAL_STATE),
              ("정상 회로", circuit_state(qaoa_p1(gamma, beta), "qaoa")),
              ("Cost만 적용", after_cost), ("Mixer 다음 cost", reversed_state)]
    rows, results = [], []
    for label, state in states:
        probabilities = state.probabilities()
        dz = float(np.max(np.abs(probabilities - 1/16)))
        mean = float(np.real(state.expectation_value(cost_operator())))
        fidelity = float(state_fidelity(INITIAL_STATE, state))
        results.append({"label": label, "state": state, "probabilities": probabilities,
                        "D_Z": dz, "mean_cut": mean, "fidelity_to_initial": fidelity})
        rows.append([label, dz, mean, fidelity])
    show_table(["비교 대상", "D_Z", "정확한 평균 점수", "초기 상태와의 F"], rows)
    show_table(["비트열", *[label for label, state in states]],
               [[bits, *[float(state.probabilities()[i]) for label, state in states]] for i, bits in enumerate(BITSTRINGS)])
    beta_zero = circuit_state(qaoa_p1(gamma, 0.0), "qaoa")
    print("Cost만 적용한 상태와 beta=0 회로의 fidelity:", float(state_fidelity(after_cost, beta_zero)))
    print("F의 기준은 초기 |+>^4 상태입니다. 최적해 확률과 구분하세요.")
    return {"parameters": (gamma, beta), "rows": results}

if RUN_OBSERVATIONS:
    _ = run_demo("3c", experiment_3c, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("3c", None)
    print("[대기] 3(c)의 삭제·순서 변경 결과를 먼저 예상하세요.")

## 문제 4. 어떤 출력 분포가 더 좋은가요?

앞 문제에서는 회로의 평균 점수를 높이는 각도를 찾고, cost와 mixer의 역할을 확인했습니다. 이번에는 <strong>그 출력 분포를 어떤 기준으로 평가할지</strong> 살펴보겠습니다.

평균 점수가 높다는 것과 최적해를 자주 얻는다는 것은 같은 뜻이 아닙니다. 또 유한한 횟수의 실행에서 최적해가 한 번 나왔다는 사실만으로, 그 회로의 최적해 확률을 알 수는 없습니다.

이번 문제에서는 평균 점수와 최적해 확률을 따로 계산하는 함수를 작성하고, 정확한 분포와 측정 기록을 같은 기준으로 평가하겠습니다.

---

### 평균 점수는 모든 답의 점수를 반영합니다

출력 비트열 $z$의 확률이 $P(z)$, 그 점수가 $C(z)$일 때 평균 점수는

$$
\langle C\rangle
=
\sum_z P(z)C(z)
$$

입니다.

최적해가 아닌 비트열도 평균에 기여합니다. 높은 점수의 여러 답이 자주 나오면, 최적해만 출력하지 않더라도 평균 점수가 높을 수 있습니다.

앞에서 사용한 고전 최적화기는 이 평균 점수에 음수를 붙인 값을 최소화했습니다. 따라서 <strong>직접 개선하려고 한 지표는 평균 점수</strong>입니다. 다른 성능 지표도 함께 좋아졌는지는 별도로 계산해서 확인해야 합니다.

---

### 최적해 확률은 최적해 집합 전체를 봅니다

문제 1에서 구한 최대 점수를 $C_{\max}$, 최적 비트열의 집합을

$$
\mathcal Z_{\mathrm{opt}}
=
\{z:C(z)=C_{\max}\}
$$

라고 하겠습니다.

최적해를 얻을 확률은

$$
P_{\mathrm{opt}}
=
\sum_{z\in\mathcal Z_{\mathrm{opt}}}P(z)
$$

입니다. 점수가 최대인 모든 비트열의 확률을 더합니다.

여기서 다음 세 가지는 서로 다릅니다.

| 확인하는 값 | 의미 |
|---|---|
| 평균 점수 $\langle C\rangle$ | 여러 번 실행했을 때 평균적으로 얻는 점수 |
| 최적해 확률 $P_{\mathrm{opt}}$ | 한 번의 실행에서 최적해 집합에 속하는 답을 얻을 확률 |
| 관찰한 최고 점수 | 실제로 얻은 유한한 기록 가운데 가장 높은 점수 |

또한 가장 자주 나온 비트열 하나의 확률이 곧 $P_{\mathrm{opt}}$인 것도 아닙니다. 그 비트열이 최적해인지 확인해야 하고, 다른 최적 비트열의 확률도 함께 더해야 합니다.

이번 그래프에서는 두 그룹의 이름을 서로 바꾼 문자열들도 같은 점수를 갖습니다. 측정 결과에서는 두 문자열이 별도의 결과로 기록되지만, 둘 다 최적해라면 둘의 확률을 모두 성공 확률에 포함합니다.

---

### 평균 점수와 최적해 확률의 순위는 다를 수 있습니다

두 회로의 출력 분포를 비교할 때에는 어떤 목적을 기준으로 삼는지 먼저 정해야 합니다.

예를 들어 모든 실행에서 얻은 점수를 누적하는 상황이라면 평균 점수가 중요한 기준입니다. 반면 최대 점수의 답을 얻어야 하는 상황이라면 최적해 확률이 중요한 기준이 됩니다.

어떤 분포는 최적해에 조금 못 미치는 답들을 자주 출력할 수 있습니다. 다른 분포는 최적해를 더 자주 출력하는 대신, 낮은 점수의 답도 더 자주 출력할 수 있습니다. 따라서 두 지표가 서로 다른 순위를 줄 수 있습니다.

이번에는 같은 그래프와 같은 $p=1$ 회로에서, 각도만 다른 두 실행을 직접 비교하겠습니다. <strong>회로 구조가 같더라도 어느 성능 지표를 선택하는지에 따라 평가가 달라질 수 있는지</strong> 확인하는 것이 목표입니다.

---

### 확률과 counts를 같은 방식으로 처리하기

정확한 상태 계산에서는 비트열별 확률을 얻습니다. 실제 측정 모의실험에서는 각 비트열이 나온 횟수인 counts를 얻습니다.

두 경우를 같은 함수로 처리하기 위해, 입력을 비트열별 <strong>가중치</strong>로 생각하겠습니다. 비트열 $z$에 주어진 가중치를 $w_z$, 전체 가중치의 합을

$$
W=\sum_z w_z
$$

라고 하면 정규화한 분포는

$$
P_w(z)=\frac{w_z}{W}
$$

입니다.

입력이 정확한 확률이라면 보통 $W=1$이고, counts라면 $W$는 전체 측정 횟수입니다.

예를 들어 다음 기록을 생각해 보겠습니다.

```python
{"0011": 6, "0000": 2}
```

전체 횟수는 8이므로, 정규화한 두 값은 각각 $6/8$과 $2/8$입니다. 문제 1의 점수를 사용하면 평균 점수는

$$
\frac{6C(0011)+2C(0000)}8
$$

로 계산합니다.

최적해 확률을 계산할 때에도 전체 가중치 $W$로 나누되, 분자에는 <strong>전체 문제의 최적해 집합에 속하는 비트열의 가중치만</strong> 더합니다.

측정 기록에 나타난 비트열 가운데 가장 높은 점수를 $C_{\max}$로 사용하면 안 됩니다. 유한한 기록에는 실제 최적해가 한 번도 나타나지 않을 수도 있기 때문입니다. 최대 점수와 최적해 집합은 문제 1에서 조사한 전체 후보를 기준으로 정합니다.

입력 딕셔너리에 없는 비트열은 가중치가 0인 것으로 처리합니다.

---

### 표본값도 점수의 범위 안에 있어야 합니다

각도를 고정하고 회로를 $N$회 실행하여

$$
z^{(1)},z^{(2)},\ldots,z^{(N)}
$$

을 얻었다고 하겠습니다. 표본 평균 점수는

$$
\widehat{\langle C\rangle}
=
\frac1N\sum_{k=1}^{N}C\bigl(z^{(k)}\bigr)
$$

입니다.

같은 회로를 사용해도 기록이 달라질 수 있으므로, 이 값은 정확한 평균 점수와 조금 다를 수 있습니다. 최적해의 관찰 비율도 정확한 $P_{\mathrm{opt}}$와 달라질 수 있습니다.

다만 이번에는 각 시행에서 얻은 비트열의 점수가 항상

$$
0\le C\bigl(z^{(k)}\bigr)\le C_{\max}
$$

이므로, 그 평균도 반드시

$$
0\le\widehat{\langle C\rangle}\le C_{\max}
$$

범위 안에 있습니다. 최적해의 관찰 비율 역시 0과 1 사이입니다.

Lab4에서는 서로 다른 측정 설정에서 추정한 여러 Pauli 기댓값을 합쳐 에너지를 구했습니다. 이번에는 <strong>한 번의 Z 측정에서 얻은 유효한 비트열에 점수를 매기고, 그 점수들을 평균</strong>합니다. 따라서 표본 평균이 가능한 최대 점수를 넘었다면, 이를 유한 shots의 변동만으로 설명할 수는 없습니다. 점수 계산과 정규화, 기록의 해석을 점검해야 합니다.

---

### 같은 조건에서 비교하기

회로를 비교할 때에는 그래프와 점수의 정의를 같게 유지합니다. 표본 결과를 비교한다면 측정 횟수도 함께 기록해야 합니다.

이번 기본 실험에서는 `cost_scale=1.0`을 유지하고, 각 회로를 1,024 shots로 실행합니다. 비교할 각도를 정한 뒤에는 측정 결과에 따라 다시 최적화하지 않습니다. 정확한 분포의 성능과 그 분포에서 얻은 표본 기록을 비교하는 실험입니다.

정확값은 회로가 정의하는 분포의 지표이고, 표본값은 그 분포를 유한한 기록으로 추정한 값입니다. 두 값이 소수점까지 같아야 한다고 판단하지 말고, 어떤 값과 어떤 기록을 비교했는지 명확하게 적으세요.

---

### 직접 풀어보기

### (a) 두 각도의 출력 분포 비교하기

다음 두 각도로 정상 $p=1$ QAOA 회로를 준비하겠습니다.

$$
(\gamma_A,\beta_A)
=
\left(\frac{\pi}{6},\frac{\pi}{16}\right),
$$

$$
(\gamma_B,\beta_B)
=
\left(\frac{\pi}{4},\frac{\pi}{8}\right).
$$

그래프, 초기 상태, 회로 구조와 `cost_scale=1.0`은 같습니다. 각도만 다릅니다.

제공된 비교 셀은 두 회로의 정확한 비트열별 확률을 보여 줍니다. 문제 1의 점수표와 최적해 집합을 사용하여 다음 표를 완성하세요.

| 실행 | 평균 점수 $\langle C\rangle$ | 최적해 확률 $P_{\mathrm{opt}}$ |
|---|---:|---:|
| A | … | … |
| B | … | … |

평균 점수는 비트열의 점수에 확률을 곱해 더하고, 최적해 확률은 최적해 집합에 속하는 확률을 모두 더해 구합니다. 평균 점수는 앞 문제의 `mean_cut_from_circuit` 결과와도 대조할 수 있습니다.

계산한 뒤 다음을 설명하세요.

1. 평균 점수가 더 높은 실행은 어느 쪽인가요?
2. 최적해를 얻을 확률이 더 높은 실행은 어느 쪽인가요?
3. 매 실행의 점수를 누적하는 상황과, 최적해를 얻어야 하는 상황에서 어느 실행을 선택하겠나요?
4. 앞의 고전 최적화기가 직접 개선하려고 한 지표는 무엇이었나요? 그 지표의 개선만으로 다른 지표도 개선되었다고 판단할 수 있나요?

두 지표의 순위가 다르다면, 어느 계산이 틀렸다고 바로 판단하지 말고 두 지표가 각각 어떤 비트열의 정보를 반영하는지 설명하세요.

---

### 4a 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 4a

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4a():
    score_table, maximum, optimal = classical_reference(score_function=cut_score)
    results = {}
    for label, parameters in (("A", (np.pi/6, np.pi/16)), ("B", (np.pi/4, np.pi/8))):
        circuit = qaoa_p1(*parameters, cost_scale=1.0)
        results[label] = {"parameters": parameters, "probabilities": probabilities_from_circuit(circuit),
                          "operator_mean": mean_cut_from_circuit(circuit)}
    show_table(["비트열", "학생 점수", "A의 확률", "B의 확률"],
               [[bits, score_table[bits], results["A"]["probabilities"][bits], results["B"]["probabilities"][bits]] for bits in BITSTRINGS])
    show_table(["실행", "gamma,beta", "관측량으로 계산한 평균"],
               [[name, row["parameters"], row["operator_mean"]] for name, row in results.items()])
    print("최적해 확률은 문제 1에서 구한 전체 최적해 집합에 속하는 확률들을 합해 먼저 계산하세요.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4a", experiment_4a, CHECK_LABELS[:3])
else:
    LAB_RESULTS.pop("4a", None)
    print("[대기] 4(a): 두 성능 지표를 비교할 준비가 되면 관찰을 실행하세요.")

### (b) 출력 분포의 두 지표를 계산하는 함수 작성하기

다음 함수를 작성하세요.

```python
distribution_metrics(weights, cost_scale=1.0)
```

`weights`는 비트열을 키로 하고 확률 또는 측정 횟수를 값으로 갖는 딕셔너리입니다.

입력은 다음 조건을 만족한다고 보장합니다.

- 키는 `z3z2z1z0` 순서의 유효한 네 비트 문자열입니다.
- 값은 유한한 음이 아닌 수입니다.
- 전체 가중치의 합은 양수입니다.
- 일부 비트열은 딕셔너리에 없을 수 있습니다.

`cost_scale`은 입력 기록의 가중치가 아니라, <strong>분할의 점수 전체에 곱하는 양의 배율</strong>입니다. 배율을 $s$라고 하면 사용하는 점수는

$$
C_s(z)=sC(z)
$$

입니다. 현재 기본 실험에서는 $s=1$을 사용하고, 마지막 적용에서 다른 배율을 사용합니다.

두 입력의 역할을 구분하세요.

| 입력 | 의미 |
|---|---|
| `weights` | 각 비트열에 주어진 확률 또는 관찰 횟수 |
| `cost_scale` | 각 분할의 점수에 곱할 공통 배율 |

비교 기준은 제공 도구로 받을 수 있습니다.

```python
score_table, maximum, optimal_strings = classical_reference(cost_scale)
```

이 도구는 전체 16개 비트열에 대해 다음을 반환합니다.

- `score_table`: 비트열별 점수 $C_s(z)$
- `maximum`: 전체 후보의 최대 점수
- `optimal_strings`: 최대 점수를 갖는 모든 비트열

함수는 다음 두 값을 실수로 담은 tuple을 반환해야 합니다.

```python
(mean_cut, p_opt)
```

- `mean_cut`: 입력 가중치를 전체 합으로 정규화하여 계산한 평균 점수
- `p_opt`: 전체 문제의 최적해 집합에 속하는 비트열들의 정규화된 가중치 합

특히 다음 조건을 확인하세요.

1. 입력이 확률일 때와 counts일 때 모두 같은 규칙을 사용합니다.
2. 없는 비트열의 가중치는 0으로 처리합니다.
3. 최적해 하나만 골라 확률을 더하지 않습니다.
4. 입력 기록에 등장한 비트열만으로 최대 점수를 다시 정하지 않습니다.

확인 셀은 정확한 확률, 정수 counts, 일부 비트열만 포함한 기록과 여러 점수 배율에서 결과를 검사합니다.

함수를 완성한 뒤 (a)의 두 분포를 입력하여, 손으로 계산한 평균 점수와 최적해 확률이 일치하는지 확인하세요.

---

In [ ]:
def distribution_metrics(weights, cost_scale=1.0):
    # TODO BEGIN — 전체 가중치와 정확한 최적해 집합으로 두 지표를 계산하세요.
    raise NotImplementedError('distribution_metrics')
    # TODO END

In [ ]:
_ = check_distribution_metrics(distribution_metrics)

### 4b 답안

계산·관찰 결과와 근거:

…

### 관찰 실험 4b

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4b():
    if "4a" not in LAB_RESULTS:
        raise ValueError("먼저 4(a)의 확률 표를 실행하세요.")
    rows, results = [], {}
    for name, data in LAB_RESULTS["4a"].items():
        mean, optimal = distribution_metrics(dict(data["probabilities"]), cost_scale=1.0)
        rows.append([name, data["parameters"], mean, optimal])
        results[name] = {"mean_cut": float(mean), "p_opt": float(optimal)}
    show_table(["실행", "gamma,beta", "학생 함수의 평균 점수", "학생 함수의 최적해 확률"], rows)
    print("4(a)의 손계산과 대조하고, 어떤 지표를 기준으로 비교했는지 적으세요.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4b", experiment_4b, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4b", None)
    print("[대기] 4(b): 지표 함수 검사 후 4(a)의 계산과 대조하세요.")

### (c) 유한한 측정 기록과 결과 보고서 검토하기

다음 세 경우를 비교하겠습니다.

- 균등 분포
- 문제 3의 초기 각도 $(0.1,0.1)$를 사용한 회로
- 문제 3에서 최적화기가 반환한 각도를 사용한 회로

제공된 측정 실험은 각 경우에 대해 1,024 shots의 기록을 만듭니다. 정확한 확률과 counts를 각각 자신의 `distribution_metrics`에 입력하여 표를 완성하세요.

| 비교 대상 | 정확한 평균 점수 | 표본 평균 점수 | 정확한 최적해 확률 | 표본 최적해 비율 | 관찰한 최고 점수 |
|---|---:|---:|---:|---:|---:|
| 균등 분포 | … | … | … | … | … |
| 초기 각도의 회로 | … | … | … | … | … |
| 최적화 후 회로 | … | … | … | … | … |

최적화 후 회로의 기록에서는 가장 자주 나온 비트열 하나를 고르고, 그 비트열의 점수와 관찰 비율도 적으세요. 가장 많은 횟수가 같은 비트열이 여러 개라면 그중 하나를 선택해도 됩니다.

이어서 다음 보고서를 검토하세요. 각 보고서에서 어떤 정보가 충분하고 어떤 판단이 성급한지 설명합니다.

> <strong>보고서 A</strong>  
> “이번 회로에서 최적해가 관찰되었으므로, 비트열을 균등하게 뽑는 방법보다 성능이 좋다.”

관찰한 최고 점수와 분포의 성능을 구분하세요. 균등 분포의 결과와 어떤 지표를 비교해야 하는지 설명합니다.

> <strong>보고서 B</strong>  
> “평균 점수가 더 높은 실행은 최적해를 얻을 확률도 더 높다.”

(a)의 두 실행을 근거로 검토하세요. 평균 점수와 최적해 확률이 반영하는 정보를 각각 설명합니다.

> <strong>보고서 C</strong>  
> “가장 자주 나온 비트열의 관찰 비율을 이 회로의 최적해 확률로 보고하겠다.”

그 비트열이 최적해인지, 다른 최적 비트열은 없는지 확인하세요. 올바른 표본 최적해 비율을 계산하려면 어떤 기록들을 합해야 하나요?

> <strong>보고서 D</strong>  
> “표본 평균 점수가 $C_{\max}$보다 높게 나왔지만, shots가 유한해서 생긴 변동일 것이다.”

이번 점수 추정이 어떤 값들의 평균인지 근거로 판단하세요. Lab4의 에너지 추정과 이번 비트열 점수 평균의 차이도 함께 설명합니다.

마지막으로 회로의 성능을 보고할 때 함께 기록할 항목을 정리하세요. 적어도 <strong>그래프와 점수의 정의, 사용한 각도, 정확값과 표본값의 구분, shots, 비교한 성능 지표</strong>가 드러나도록 작성합니다.

---

### 4c 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 관찰 실험 4c

해당 문항의 예상과 계산을 먼저 적은 뒤 실행하세요. 필요한 함수 검사를 통과하면 실제 학생 함수와 제공 도구로 결과를 확인합니다. 결과가 대기로 표시되면 앞의 관찰 설정과 확인 셀을 점검하세요.

In [ ]:
def experiment_4c():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b)의 최적화를 실행하세요.")
    uniform = QuantumCircuit(4)
    uniform.h(range(4))
    circuits = [("균등 분포", uniform), ("초기 각도", qaoa_p1(0.1, 0.1)),
                ("최적화 후", qaoa_p1(*LAB_RESULTS["3b"]["parameters"]))]
    results, rows = [], []
    for index, (label, circuit) in enumerate(circuits):
        probabilities = probabilities_from_circuit(circuit)
        counts = sample_counts(circuit, shots=1024, seed=20261109+index)
        exact_mean, exact_opt = distribution_metrics(dict(probabilities), cost_scale=1.0)
        sample_mean, sample_opt = distribution_metrics(dict(counts), cost_scale=1.0)
        best = max(cut_score(bits) for bits in counts)
        mode = max(counts, key=lambda bits: (counts[bits], bits))
        results.append({"label": label, "probabilities": probabilities, "counts": counts,
                        "exact_mean": float(exact_mean), "exact_p_opt": float(exact_opt),
                        "sample_mean": float(sample_mean), "sample_p_opt": float(sample_opt),
                        "observed_best": best, "mode": mode, "mode_score": cut_score(mode),
                        "mode_frequency": counts[mode]/1024, "shots": 1024})
        rows.append([label, exact_mean, sample_mean, exact_opt, sample_opt, best])
    show_table(["비교 대상", "정확 평균", "표본 평균", "정확 최적해 확률", "표본 최적해 비율", "관찰 최고 점수"], rows)
    show_table(["비트열", *[row["label"] for row in results]],
               [[bits, *[row["counts"].get(bits, 0) for row in results]] for bits in BITSTRINGS])
    show_table(["비교 대상", "최빈 비트열 하나", "그 점수", "관찰 비율"],
               [[row["label"], row["mode"], row["mode_score"], row["mode_frequency"]] for row in results])
    print("모든 경우 1,024 shots입니다. 최빈 횟수가 같으면 문자열 순서로 하나를 선택했습니다.")
    return results

if RUN_OBSERVATIONS:
    _ = run_demo("4c", experiment_4c, CHECK_LABELS)
else:
    LAB_RESULTS.pop("4c", None)
    print("[대기] 4(c): 정확 분포와 같은 조건의 표본 기록을 비교하세요.")

## 마지막 적용 — 모든 간선의 점수가 두 배가 된다면?

이제 그래프의 연결, 초기 상태와 회로 구조는 유지하고, 모든 간선의 점수를 1에서 2로 바꾸겠습니다.

$$
C'(z)=2C(z),
\qquad
\widehat C'=2\widehat C.
$$

이번 변경에서는 출력에 점수를 매기는 기준뿐 아니라, <strong>회로의 cost 연산에도 새 점수 연산자를 사용</strong>합니다. 코드에서는 `cost_scale=2.0`으로 지정합니다.

앞의 최적화에서 찾은 각도를 $(\gamma_*,\beta_*)$라고 하겠습니다. 같은 각도를 그대로 사용하는 경우와, 원래 출력 분포를 유지하도록 각도를 수정한 경우를 비교하겠습니다.

먼저 다음을 예상하세요.

1. 점수의 최대값과 최적해 집합은 어떻게 달라질까요?
2. `cost_scale`만 2.0으로 바꾸고 각도를 그대로 두면, cost 연산이 상태에 붙이는 위상은 이전과 같을까요?
3. 원래 출력 분포를 유지하려면 $\gamma,\beta$를 어떻게 조절하면 좋을까요?

각도 수정은 새로운 최적화를 실행하기 전에 제안하세요. 문제 2에서 사용한

$$
U_C(\gamma;s)=e^{-i\gamma s\widehat C}
$$

의 정의를 근거로 설명합니다.

그다음 제공된 비교 도구로 다음 세 경우의 정확한 분포를 구합니다.

| 비교 대상 | `cost_scale` | 사용할 각도 |
|---|---:|---|
| 원래 실험 | 1.0 | $(\gamma_*,\beta_*)$ |
| 점수 배율만 변경 | 2.0 | $(\gamma_*,\beta_*)$ |
| 점수 배율과 각도 수정 | 2.0 | 자신이 제안한 각도 |

각 분포를 평가할 때에는 `distribution_metrics`에도 해당 `cost_scale`을 전달하세요. 회로에서 사용한 점수와 결과를 평가하는 점수가 일치해야 합니다.

원래 분포를 $P_{\mathrm{original}}(z)$라고 하면, 제공 도구는 각 비교 분포와의 차이

$$
D_{\mathrm{original}}
=
\max_z
\left|P(z)-P_{\mathrm{original}}(z)\right|
$$

도 계산합니다. 이 비교에는 표본 횟수가 아니라 정확한 확률을 사용합니다.

| 비교 대상 | 해당 점수 단위의 평균 점수 | 해당 문제의 최대 점수 | 최적해 확률 | $D_{\mathrm{original}}$ |
|---|---:|---:|---:|---:|
| 원래 실험 | … | … | … | … |
| 점수 배율만 변경 | … | … | … | … |
| 점수 배율과 각도 수정 | … | … | … | … |

결과를 확인한 뒤 다음을 설명하세요.

- 제안한 각도 수정으로 원래 출력 분포를 유지했나요? 정의식과 확률 비교가 같은 결론을 주나요?
- 분포를 유지한 경우에도 평균 점수의 숫자는 어떻게 달라지나요? 최대 점수와 최적해 확률은 각각 어떻게 비교되나요?
- “평균 점수가 두 배가 되었으므로 알고리즘 성능도 두 배 좋아졌다”는 설명이 적절한가요?

새로운 회로나 최적화기를 작성할 필요는 없습니다. <strong>점수의 단위가 달라진 것과 더 좋은 출력 분포를 만든 것을 구분</strong>하고, 어떤 조건을 유지하기 위해 어떤 매개변수를 수정했는지 정리하세요.

## 마지막 적용 답안

실행 전 예상과 계산:

…

관찰 결과와 설명:

…

### 적용 결과 확인

`APPLICATION_PARAMETERS`에는 제안할 `(gamma, beta)`를 적으세요. `APPLICATION_PREDICTIONS`에는 `maximum`(새 최대 점수), `optimal_strings`(새 최적 비트열들의 tuple), `same_angles_keep_distribution`(같은 각도에서 분포 유지 여부, True 또는 False)의 세 키를 갖는 dict를 입력합니다. 예상과 이유를 적은 뒤 `RUN_APPLICATION=True`로 바꾸고 관찰 셀을 실행하세요. 이 변수들은 새 함수 TODO가 아닙니다.

In [ ]:
RUN_APPLICATION = False
APPLICATION_PARAMETERS = None
APPLICATION_PREDICTIONS = None

In [ ]:
def application_experiment():
    if "3b" not in LAB_RESULTS:
        raise ValueError("먼저 3(b)의 최적화를 실행하세요.")
    if APPLICATION_PARAMETERS is None or APPLICATION_PREDICTIONS is None:
        raise ValueError("제안 각도와 세 가지 예측을 먼저 입력하세요.")
    proposed = tuple(map(float, APPLICATION_PARAMETERS))
    if len(proposed) != 2 or not np.isfinite(proposed).all():
        raise ValueError("제안 각도는 유한한 gamma,beta 두 값입니다.")
    predictions = APPLICATION_PREDICTIONS
    if not isinstance(predictions, dict) or not {'maximum', 'optimal_strings', 'same_angles_keep_distribution'} <= set(predictions):
        raise ValueError("예측 키 maximum, optimal_strings, same_angles_keep_distribution을 확인하세요.")
    if not isinstance(predictions['same_angles_keep_distribution'], (bool, np.bool_)):
        raise ValueError("분포 유지 예상은 True 또는 False입니다.")
    baseline = LAB_RESULTS["3b"]["parameters"]
    original = weight_vector(probabilities_from_circuit(qaoa_p1(*baseline, cost_scale=1.0)))
    results, rows = [], []
    for label, parameters, scale in (("원래 실험", baseline, 1.0),
                                     ("배율만 변경", baseline, 2.0),
                                     ("배율과 각도 수정", proposed, 2.0)):
        circuit = qaoa_p1(*parameters, cost_scale=scale)
        probabilities = probabilities_from_circuit(circuit)
        mean, optimal_probability = distribution_metrics(dict(probabilities), cost_scale=scale)
        score_table, maximum, optimal_strings = classical_reference(scale, score_function=cut_score)
        difference = float(np.max(np.abs(weight_vector(probabilities)-original)))
        rows.append([label, scale, parameters, mean, maximum, optimal_probability, difference])
        results.append({"label": label, "cost_scale": scale, "parameters": parameters,
                        "probabilities": probabilities, "mean_cut": float(mean),
                        "maximum": maximum, "optimal_strings": optimal_strings,
                        "p_opt": float(optimal_probability), "distribution_difference": difference})
    show_table(["비교 대상", "배율", "각도", "해당 단위 평균", "최대 점수", "최적해 확률", "원래 분포와의 차이"], rows)
    same_angles_preserve = results[1]["distribution_difference"] <= ATOL
    checks = {"maximum": bool(np.isclose(float(predictions['maximum']), results[1]['maximum'], atol=ATOL, rtol=0)),
              "optimal_strings": set(predictions['optimal_strings']) == set(results[1]['optimal_strings']),
              "same_angles_keep_distribution": bool(predictions['same_angles_keep_distribution']) == same_angles_preserve}
    show_table(["예측 항목", "입력한 예상", "실제 결과", "일치"],
               [["새 최대 점수", predictions['maximum'], results[1]['maximum'], checks['maximum']],
                ["새 최적해 집합", tuple(predictions['optimal_strings']), results[1]['optimal_strings'], checks['optimal_strings']],
                ["기존 각도로 분포 유지", predictions['same_angles_keep_distribution'], same_angles_preserve, checks['same_angles_keep_distribution']]])
    preserved = results[2]["distribution_difference"] <= ATOL
    print("제안한 각도의 원래 분포 유지:", preserved)
    print("분포 보존을 비교하며, 상태 fidelity=1을 별도의 성공 조건으로 요구하지 않습니다.")
    return {"rows": results, "prediction_checks": checks, "predictions_match": all(checks.values()),
            "distribution_preserved": preserved}

LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment, CHECK_LABELS)
else:
    print("[대기] 마지막 적용: 예상과 제안 각도를 작성한 뒤 별도 실행 설정을 켜세요.")

## Lab 5 마무리

네 코드 검사의 상태와 각 소문항의 설명을 확인하세요. 점수 함수가 관측량과 위상 연산으로 연결되는 과정, cost와 mixer의 역할, 평균 점수와 최적해 확률의 차이, 점수 단위 변경과 출력 분포 개선의 차이를 세 문장 또는 네 문장으로 정리하세요.

### 마무리 답안

계산·관찰 결과와 근거:

…

In [ ]:
expected_checks = ['1b · 분할 점수', '2b · cost 연산', '2c · QAOA 한 층', '4b · 분포 지표']
show_table(["코드 확인", "상태"], [[name, CHECK_STATUS.get(name, "미실행")] for name in expected_checks])
if all(CHECK_STATUS.get(name) == "통과" for name in expected_checks):
    print("네 코드 검사를 모두 통과했습니다. 서술 답안과 마지막 적용 해석은 별도로 확인합니다.")
else:
    print("TODO와 확인 셀을 작성·실행한 뒤 연결 실험도 다시 실행하세요.")